# AlphaGPT × Gate BTC：最佳公式挖掘

依次运行所有单元。建议在「运行时 → 更改运行时类型」选择 T4 GPU。
本 Notebook 内嵌当前仓库的 AlphaGPT 生成器和 BTC 适配代码，无需克隆仓库。

**市场**：Gate BTC_USDT USDT 永续，1 小时，做多/做空/空仓，1 倍目标名义敞口。
公开行情与历史资金费率无需 API Key；没有交易或私钥代码。

最佳 = 训练集排名前 K 个公式中，验证集净收益风险评分最高的候选。
测试集只在选定公式后评估一次；负收益也如实导出，不宣称找到盈利策略。

Gate 接口文档：https://www.gate.com/docs/developers/apiv4/en/futures/


In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "requests>=2.32,<3" "matplotlib>=3.9,<4" "plotly>=5.24,<7" "pillow>=10,<13"
import torch
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.is_available())
# 保留 Colab 自带的 GPU 版 PyTorch；不要安装仓库全部 Solana/数据库依赖。


## 1. 写入内嵌源码（可展开查看）

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path("/content/alphagpt_btc")
ROOT.mkdir(parents=True, exist_ok=True)
FILES = {'model_core/alphagpt.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom .config import ModelConfig\nfrom .vocab import FORMULA_VOCAB\n\n\nclass NewtonSchulzLowRankDecay:\n    """\n    Low-Rank Decay (LoRD) using Newton-Schulz iteration.\n    \n    A more efficient regularization method that targets low-rank structure\n    in attention and key parameters. Uses Newton-Schulz iteration to compute\n    the minimum singular vectors without explicit SVD.\n    \n    Args:\n        named_parameters: Model\'s named parameters\n        decay_rate: Strength of low-rank decay\n        num_iterations: Number of Newton-Schulz iterations (default: 5)\n        target_keywords: If specified, only decay parameters matching these keywords\n    """\n    def __init__(self, named_parameters, decay_rate=1e-3, num_iterations=5, target_keywords=None):\n        self.decay_rate = decay_rate\n        self.num_iterations = num_iterations\n        self.target_keywords = target_keywords or ["qk_norm", "attention"]\n        self.params_to_decay = []\n        \n        for name, param in named_parameters:\n            if not param.requires_grad or param.ndim != 2:\n                continue\n            if not any(k in name for k in self.target_keywords):\n                continue\n            self.params_to_decay.append((name, param))\n    \n    @torch.no_grad()\n    def step(self):\n        """Apply Newton-Schulz low-rank decay to attention parameters."""\n        for name, W in self.params_to_decay:\n            orig_dtype = W.dtype\n            X = W.float()\n            r, c = X.shape\n            \n            # Transpose if needed for efficiency\n            transposed = False\n            if r > c:\n                X = X.T\n                transposed = True\n            \n            # Normalize by spectral norm\n            norm = X.norm() + 1e-8\n            X = X / norm\n            \n            # Initialize Y for Newton-Schulz iteration\n            Y = X\n            I = torch.eye(X.shape[-1], device=X.device, dtype=X.dtype)\n            \n            # Newton-Schulz iteration: Y_{k+1} = 0.5 * Y_k * (3*I - Y_k^T * Y_k)\n            # This converges to the orthogonal matrix with same singular vectors\n            for _ in range(self.num_iterations):\n                A = Y.T @ Y\n                Y = 0.5 * Y @ (3.0 * I - A)\n            \n            if transposed:\n                Y = Y.T\n            \n            # Apply low-rank decay\n            W.sub_(self.decay_rate * Y.to(orig_dtype))\n\n\nclass StableRankMonitor:\n    """Monitor the effective rank (stable rank) of model parameters."""\n    def __init__(self, model, target_keywords=None):\n        self.model = model\n        self.target_keywords = target_keywords or ["q_proj", "k_proj", "attention"]\n        self.history = []\n    \n    @torch.no_grad()\n    def compute(self):\n        """Compute average stable rank of target parameters."""\n        ranks = []\n        for name, param in self.model.named_parameters():\n            if param.ndim != 2:\n                continue\n            if not any(k in name for k in self.target_keywords):\n                continue\n            \n            W = param.detach().float()\n            S = torch.linalg.svdvals(W)\n            # Stable Rank = ||W||_F^2 / ||W||_2^2\n            stable_rank = (S.norm() ** 2) / (S[0] ** 2 + 1e-9)\n            ranks.append(stable_rank.item())\n        \n        avg_rank = sum(ranks) / len(ranks) if ranks else 0.0\n        self.history.append(avg_rank)\n        return avg_rank\n\n\nclass RMSNorm(nn.Module):\n    """Root Mean Square Layer Normalization"""\n    def __init__(self, d_model, eps=1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(d_model))\n    \n    def forward(self, x):\n        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)\n        return (x / rms) * self.weight\n\n\nclass QKNorm(nn.Module):\n    """Query-Key Normalization for Attention"""\n    def __init__(self, d_model, eps=1e-6):\n        super().__init__()\n        self.eps = eps\n        self.scale = nn.Parameter(torch.ones(1, 1, 1, d_model) * (d_model ** -0.5))\n    \n    def forward(self, q, k):\n        # Normalize Q and K independently\n        q_norm = F.normalize(q, p=2, dim=-1)\n        k_norm = F.normalize(k, p=2, dim=-1)\n        return q_norm * self.scale, k_norm * self.scale\n\n\nclass SwiGLU(nn.Module):\n    """Swish GLU activation function"""\n    def __init__(self, d_in, d_ff):\n        super().__init__()\n        self.w = nn.Linear(d_in, d_ff * 2)\n        self.fc = nn.Linear(d_ff, d_in)\n    \n    def forward(self, x):\n        x_glu = self.w(x)\n        x, gate = x_glu.chunk(2, dim=-1)\n        x = x * F.silu(gate)  # Swish activation\n        return self.fc(x)\n\n\nclass MTPHead(nn.Module):\n    """Multi-Task Pooling Head for multi-objective learning"""\n    def __init__(self, d_model, vocab_size, num_tasks=3):\n        super().__init__()\n        self.num_tasks = num_tasks\n        self.task_heads = nn.ModuleList([\n            nn.Linear(d_model, vocab_size) for _ in range(num_tasks)\n        ])\n        self.task_weights = nn.Parameter(torch.ones(num_tasks) / num_tasks)\n        self.task_router = nn.Sequential(\n            nn.Linear(d_model, d_model // 2),\n            nn.ReLU(),\n            nn.Linear(d_model // 2, num_tasks)\n        )\n    \n    def forward(self, x):\n        # Route to appropriate task heads\n        task_logits = self.task_router(x)\n        task_probs = F.softmax(task_logits, dim=-1)\n        \n        # Compute all task outputs\n        task_outputs = [head(x) for head in self.task_heads]\n        task_outputs = torch.stack(task_outputs, dim=1)  # [B, num_tasks, vocab_size]\n        \n        # Weighted combination\n        weighted = (task_probs.unsqueeze(-1) * task_outputs).sum(dim=1)\n        return weighted, task_probs\n\n\nclass LoopedTransformerLayer(nn.Module):\n    """Looped Transformer Layer - recurrent processing within a layer"""\n    def __init__(self, d_model, nhead, dim_feedforward, num_loops=3, dropout=0.1):\n        super().__init__()\n        self.num_loops = num_loops\n        self.d_model = d_model\n        self.nhead = nhead\n        \n        # QK-Norm attention\n        self.qk_norm = QKNorm(d_model // nhead)\n        \n        # Standard attention components\n        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)\n        \n        # RMSNorm instead of LayerNorm\n        self.norm1 = RMSNorm(d_model)\n        self.norm2 = RMSNorm(d_model)\n        \n        # SwiGLU FFN instead of standard FFN\n        self.ffn = SwiGLU(d_model, dim_feedforward)\n        \n        self.dropout = nn.Dropout(dropout)\n    \n    def forward(self, x, mask=None, is_causal=False):\n        # Looped processing - recurrent refinement\n        for _ in range(self.num_loops):\n            # Self-attention with residual\n            x_norm = self.norm1(x)\n            attn_out, _ = self.attention(x_norm, x_norm, x_norm, attn_mask=mask, is_causal=is_causal)\n            x = x + self.dropout(attn_out)\n            \n            # FFN with residual\n            x_norm = self.norm2(x)\n            ffn_out = self.ffn(x_norm)\n            x = x + self.dropout(ffn_out)\n        \n        return x\n\n\nclass LoopedTransformer(nn.Module):\n    """Looped Transformer Encoder with multiple loop iterations"""\n    def __init__(self, d_model, nhead, num_layers, dim_feedforward, num_loops=3, dropout=0.1):\n        super().__init__()\n        self.layers = nn.ModuleList([\n            LoopedTransformerLayer(d_model, nhead, dim_feedforward, num_loops, dropout)\n            for _ in range(num_layers)\n        ])\n    \n    def forward(self, x, mask=None, is_causal=False):\n        for layer in self.layers:\n            x = layer(x, mask=mask, is_causal=is_causal)\n        return x\n\n\nclass AlphaGPT(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.d_model = 64\n        self.features_list = list(FORMULA_VOCAB.feature_names)\n        self.ops_list = list(FORMULA_VOCAB.operator_names)\n        \n        self.vocab = list(FORMULA_VOCAB.token_names)\n        self.vocab_size = FORMULA_VOCAB.size\n        \n        # Embedding\n        self.token_emb = nn.Embedding(self.vocab_size, self.d_model)\n        self.pos_emb = nn.Parameter(torch.zeros(1, ModelConfig.MAX_FORMULA_LEN + 1, self.d_model))\n        \n        # Enhanced Transformer with Looped Transformer\n        self.blocks = LoopedTransformer(\n            d_model=self.d_model,\n            nhead=4,\n            num_layers=2,\n            dim_feedforward=128,\n            num_loops=3,\n            dropout=0.1\n        )\n        \n        # RMSNorm instead of LayerNorm\n        self.ln_f = RMSNorm(self.d_model)\n        \n        # MTPHead for multi-task output\n        self.mtp_head = MTPHead(self.d_model, self.vocab_size, num_tasks=3)\n        self.head_critic = nn.Linear(self.d_model, 1)\n\n    def forward(self, idx):\n        # idx: [Batch, SeqLen]\n        B, T = idx.size()\n        \n        x = self.token_emb(idx) + self.pos_emb[:, :T, :]\n        \n        # Causal Mask\n        mask = nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)\n        \n        # Process through looped transformer\n        x = self.blocks(x, mask=mask, is_causal=True)\n        x = self.ln_f(x)\n        \n        last_emb = x[:, -1, :]\n        \n        # Multi-task pooling head for logits\n        logits, task_probs = self.mtp_head(last_emb)\n        value = self.head_critic(last_emb)\n        \n        return logits, value, task_probs\n', 'model_core/config.py': 'import torch\nimport os\nfrom .vocab import FORMULA_VOCAB\n\nclass ModelConfig:\n    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    DB_URL = f"postgresql://{os.getenv(\'DB_USER\',\'postgres\')}:{os.getenv(\'DB_PASSWORD\',\'password\')}@{os.getenv(\'DB_HOST\',\'localhost\')}:5432/{os.getenv(\'DB_NAME\',\'crypto_quant\')}"\n    BATCH_SIZE = 8192\n    TRAIN_STEPS = 1000\n    MAX_FORMULA_LEN = 12\n    TRADE_SIZE_USD = 1000.0\n    MIN_LIQUIDITY = 5000.0 # 低于此流动性视为归零/无法交易\n    BASE_FEE = 0.005 # 基础费率 0.5% (Swap + Gas + Jito Tip)\n    INPUT_DIM = FORMULA_VOCAB.feature_count\n', 'model_core/vocab.py': 'from dataclasses import dataclass\n\nfrom .ops import OPS_CONFIG\n\n\nFEATURE_NAMES = (\n    "RET",\n    "LIQ_SCORE",\n    "PRESSURE",\n    "FOMO",\n    "DEV",\n    "LOG_VOL",\n)\n\n\n@dataclass(frozen=True)\nclass FormulaVocab:\n    feature_names: tuple[str, ...]\n    operator_names: tuple[str, ...]\n\n    @property\n    def feature_count(self) -> int:\n        return len(self.feature_names)\n\n    @property\n    def operator_offset(self) -> int:\n        return self.feature_count\n\n    @property\n    def token_names(self) -> tuple[str, ...]:\n        return self.feature_names + self.operator_names\n\n    @property\n    def size(self) -> int:\n        return len(self.token_names)\n\n\nFORMULA_VOCAB = FormulaVocab(\n    feature_names=FEATURE_NAMES,\n    operator_names=tuple(cfg[0] for cfg in OPS_CONFIG),\n)\n', 'model_core/ops.py': "import torch\n\n@torch.jit.script\ndef _ts_delay(x: torch.Tensor, d: int) -> torch.Tensor:\n    if d == 0: return x\n    pad = torch.zeros((x.shape[0], d), device=x.device)\n    return torch.cat([pad, x[:, :-d]], dim=1)\n\n@torch.jit.script\ndef _op_gate(condition: torch.Tensor, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n    mask = (condition > 0).float()\n    return mask * x + (1.0 - mask) * y\n\n@torch.jit.script\ndef _op_jump(x: torch.Tensor) -> torch.Tensor:\n    mean = x.mean(dim=1, keepdim=True)\n    std = x.std(dim=1, keepdim=True) + 1e-6\n    z = (x - mean) / std\n    return torch.relu(z - 3.0)\n\n@torch.jit.script\ndef _op_decay(x: torch.Tensor) -> torch.Tensor:\n    return x + 0.8 * _ts_delay(x, 1) + 0.6 * _ts_delay(x, 2)\n\nOPS_CONFIG = [\n    ('ADD', lambda x, y: x + y, 2),\n    ('SUB', lambda x, y: x - y, 2),\n    ('MUL', lambda x, y: x * y, 2),\n    ('DIV', lambda x, y: x / (y + 1e-6), 2),\n    ('NEG', lambda x: -x, 1),\n    ('ABS', torch.abs, 1),\n    ('SIGN', torch.sign, 1),\n    ('GATE', _op_gate, 3),\n    ('JUMP', _op_jump, 1),\n    ('DECAY', _op_decay, 1),\n    ('DELAY1', lambda x: _ts_delay(x, 1), 1),\n    ('MAX3', lambda x: torch.max(x, torch.max(_ts_delay(x,1), _ts_delay(x,2))), 1)\n]", 'colab/btc_colab.py': '"""Compatibility entry point; authoritative implementation lives in btc_mining.core."""\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom btc_mining import core\nif __name__ == \'__main__\':\n    core.main()\nelse:\n    sys.modules[__name__] = core\n', 'colab/test_btc_colab.py': '"""Offline correctness and tiny end-to-end checks; synthetic results are not BTC evidence."""\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom btc_colab import (Config, FEATURES, OPS, TOKENS, EOS, Backtester, decode,\n                       execute, fetch_gate, make_features, make_generator, run,\n                       sample_formulas, utc_seconds, validate_candles, validate_funding)\n\n\ndef fixture(n=1800):\n    rng = np.random.default_rng(7)\n    times = utc_seconds(\'2025-01-01\') + np.arange(n) * 3600\n    price = 90000 * np.exp(np.cumsum(rng.normal(0.0001, 0.003, n)))\n    close = price * np.exp(rng.normal(0, 0.001, n))\n    df = pd.DataFrame({\'time\': times, \'open\': price,\n        \'high\': np.maximum(price, close) * 1.002, \'low\': np.minimum(price, close) * 0.998,\n        \'close\': close, \'volume\': rng.lognormal(16, 0.5, n)})\n    fund = pd.DataFrame({\'time\': times[::8], \'rate\': 0.0001})\n    cfg = Config(start=\'2025-01-01\', end=str(pd.Timestamp(times[-1]+3600, unit=\'s\', tz=\'UTC\')),\n                 steps=2, batch_size=8, seeds=(42,), norm_window=48, min_entries=1, top_k=10, charts=False)\n    return df, fund, cfg\n\n\nclass CorrectnessTests(unittest.TestCase):\n    def test_config_rejects_invalid_training_parameters(self):\n        _, _, cfg = fixture()\n        for name, value in [(\'steps\', 1.5), (\'batch_size\', 1), (\'learning_rate\', 0),\n                            (\'fee_bps\', float(\'nan\')), (\'train_fraction\', 0.9)]:\n            from dataclasses import replace\n            with self.assertRaises(ValueError): replace(cfg, **{name:value}).validate()\n\n    def test_completed_output_cannot_mix_two_runs(self):\n        df, fund, cfg = fixture()\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            (root/\'best_btc_formula.json\').write_text(\'{}\')\n            with self.assertRaisesRegex(ValueError,\'completed run\'):\n                run(cfg,root)\n\n    def test_future_changes_do_not_change_past_features_or_formulas(self):\n        df, _, cfg = fixture()\n        altered = df.copy()\n        altered.loc[900:, [\'open\',\'high\',\'low\',\'close\']] *= 3\n        altered.loc[900:, \'volume\'] *= 5\n        a, _ = make_features(df, cfg.norm_window)\n        b, _ = make_features(altered, cfg.norm_window)\n        np.testing.assert_array_equal(a[:, :900], b[:, :900])\n        for i, (_, arity) in enumerate(OPS):\n            formula = [0, 1, 2][:arity] + [len(FEATURES)+i]\n            np.testing.assert_array_equal(execute(formula, a)[:900], execute(formula, b)[:900])\n\n    def test_prefix_matches_full_history(self):\n        df, _, cfg = fixture()\n        all_f, _ = make_features(df, cfg.norm_window)\n        short_f, _ = make_features(df.iloc[:900], cfg.norm_window)\n        np.testing.assert_array_equal(all_f[:, :900], short_f)\n\n    def test_grid_rejects_missing_or_invalid_candles(self):\n        df, _, cfg = fixture()\n        self.assertEqual(len(validate_candles(df, cfg)), len(df))\n        with self.assertRaises(ValueError): validate_candles(df.drop(index=3), cfg)\n        df.loc[3, \'low\'] = df.loc[3, \'high\'] * 2\n        with self.assertRaises(ValueError): validate_candles(df, cfg)\n\n    def test_funding_coverage(self):\n        _, funding, cfg = fixture()\n        self.assertEqual(len(validate_funding(funding, cfg)), len(funding))\n        with self.assertRaises(ValueError): validate_funding(funding.iloc[10:], cfg)\n\n    def test_stack_and_protected_division(self):\n        f = np.zeros((6, 30)); f[0] = 2; f[1] = 3\n        self.assertEqual(decode([0,1,7]), \'SUB(RET, MOM24)\')\n        np.testing.assert_array_equal(execute([0,1,7], f), -np.ones(30))\n        self.assertTrue(np.isfinite(execute([0,2,9], f)).all())\n        with self.assertRaises(ValueError): execute([6], f)\n        with self.assertRaises(ValueError): execute([0,1], f)\n\n    def test_sampler_always_emits_valid_formulas_and_gradients(self):\n        torch.set_num_threads(2); torch.manual_seed(5)\n        model = make_generator(\'cpu\')\n        for length in [2, 5, 12]:\n            tokens, logp = sample_formulas(model, 32, length, \'cpu\')\n            self.assertTrue(torch.isfinite(logp).all())\n            for formula in tokens:\n                self.assertTrue(np.isfinite(execute(formula, np.ones((6, 30)))).all())\n                decode(formula)\n            model.zero_grad(); logp.mean().backward()\n            self.assertTrue(any(p.grad is not None for p in model.parameters()))\n\n    def test_execution_purge_funding_and_costs(self):\n        df, funding, cfg = fixture()\n        _, warmup = make_features(df, cfg.norm_window)\n        bt = Backtester(df, funding, cfg, warmup)\n        i = 100\n        self.assertAlmostEqual(bt.ret[i], df.open.iloc[i+2]/df.open.iloc[i+1]-1)\n        # A settlement at time index 104 is charged to signal index 102.\n        self.assertAlmostEqual(bt.funding[102], 0.0001)\n        self.assertEqual(bt.funding[103], 0)\n        a, b = bt.splits[\'train\']; c, d = bt.splits[\'validation\']\n        self.assertLess(b-1+2, c)\n        a, b = bt.splits[\'test\']\n        bt.ret[:] = 0; bt.funding[:] = 0\n        p = np.ones(b-a)\n        result = bt.evaluate_positions(p, a, b, include_series=True)\n        net = result[\'series\'].net_return.to_numpy()\n        fee = (cfg.fee_bps+cfg.slippage_bps)/10000\n        self.assertAlmostEqual(net[0], -fee)\n        self.assertLess(net[-1], -fee+1e-9)\n        np.testing.assert_array_equal(bt.evaluate_positions(np.zeros(b-a), a,b, include_series=True)[\'series\'].net_return, 0)\n        bt.funding[a:b] = 0.001\n        cfg.fee_bps = cfg.slippage_bps = 0\n        long = bt.evaluate_positions(p,a,b,include_series=True)[\'series\'].net_return\n        short = bt.evaluate_positions(-p,a,b,include_series=True)[\'series\'].net_return\n        np.testing.assert_allclose(long, -0.001)\n        np.testing.assert_allclose(short, 0.001)\n\n    def test_gate_pagination_and_quote_volume_mapping(self):\n        df, fund, cfg = fixture()\n        calls = []\n        def fake(session, url, params):\n            calls.append((url, params))\n            source = df if url.endswith(\'candlesticks\') else fund\n            part = source[(source.time >= params[\'from\']) & (source.time <= params[\'to\'])]\n            if url.endswith(\'candlesticks\'):\n                self.assertNotIn(\'limit\', params)\n                self.assertLessEqual((params[\'to\']-params[\'from\'])//3600+1, 1000)\n                return [{\'t\': int(x.time), \'o\': str(x.open), \'h\': str(x.high), \'l\': str(x.low),\n                         \'c\': str(x.close), \'sum\': str(x.volume), \'v\': \'999\'} for x in part.itertuples()]\n            return [{\'t\':int(x.time), \'r\':str(x.rate)} for x in part.itertuples()]\n        with tempfile.TemporaryDirectory() as tmp, patch(\'btc_colab.get_json\', fake), patch(\'btc_colab.time.sleep\'), patch(\'btc_colab.time.time\', return_value=utc_seconds(cfg.end)+3600):\n            got, rates, _ = fetch_gate(cfg, tmp)\n            np.testing.assert_allclose(got.volume, df.volume)\n            self.assertEqual(len(rates), len(fund))\n            count = len(calls)\n            fetch_gate(cfg, tmp)\n            self.assertEqual(len(calls), count)\n\n    def test_end_to_end_export_and_test_is_not_used_for_selection(self):\n        df, fund, cfg = fixture()\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            df.to_csv(root/\'candles.csv\', index=False); fund.to_csv(root/\'funding.csv\', index=False)\n            events = []\n            original = Backtester.evaluate\n            def record(bt, signal, split, *args, **kwargs):\n                events.append(split)\n                return original(bt, signal, split, *args, **kwargs)\n            with patch.object(Backtester, \'evaluate\', record):\n                best = run(cfg, root/\'results\', root/\'candles.csv\', root/\'funding.csv\')\n            self.assertEqual(events[-1], \'test\')\n            self.assertEqual(events.count(\'test\'), 1)\n            saved = json.loads((root/\'results/best_btc_formula.json\').read_text(encoding=\'utf-8\'))\n            self.assertEqual(saved[\'formula\'], decode(saved[\'tokens\']))\n            self.assertEqual(best[\'test\'][\'bars\'], len(pd.read_csv(root/\'results/test_equity.csv\')))\n            self.assertTrue((root/\'results/reproduction.json\').exists())\n\n\nif __name__ == \'__main__\': unittest.main()\n', 'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'btc_mining/charts.py': '"""Nine evidence-backed figures and a portable interactive HTML report."""\nimport html\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport plotly.graph_objects as go\n\nfrom .figure_tools.setup_style import setup_style\nfrom .figure_tools.export_figure import export_figure\nfrom .figure_tools.visual_qa import audit_layout\n\nBLUE, ORANGE, GREEN = \'#0072B2\', \'#D55E00\', \'#009E73\'\n\n\ndef load_results(directory):\n    root = Path(directory)\n    return (json.loads((root/\'best_btc_formula.json\').read_text(encoding=\'utf-8\')),\n            pd.read_csv(root/\'test_equity.csv\'), pd.read_csv(root/\'test_baselines.csv\'),\n            pd.read_csv(root/\'training_history.csv\'), pd.read_csv(root/\'candidates.csv\'))\n\n\ndef interactive_figures(directory):\n    best, series, baselines, history, _ = load_results(directory)\n    dates = pd.to_datetime(series.exit_time, unit=\'s\', utc=True)\n    equity = go.Figure()\n    for name, values, color, dash in [(\'Strategy\', series.equity, BLUE, \'solid\'),\n        (\'Long 1x\', baselines.long_1x, ORANGE, \'dash\'), (\'Short 1x\', baselines.short_1x, GREEN, \'dot\')]:\n        equity.add_trace(go.Scatter(x=dates, y=values, name=name, line=dict(color=color, dash=dash)))\n    equity.add_hline(y=1, line_dash=\'dot\', line_color=\'#888888\')\n    equity.update_layout(title=\'Untouched test equity · after costs and funding\', yaxis_title=\'Equity (initial = 1)\')\n    peaks = np.maximum.accumulate(np.r_[1.0, series.equity])[1:]\n    drawdown = go.Figure(go.Scatter(x=dates, y=1-series.equity/peaks, fill=\'tozeroy\',\n                                  line_color=ORANGE, name=\'Drawdown\'))\n    drawdown.update_layout(title=\'Test drawdown from running peak\', yaxis_title=\'Drawdown\', yaxis_tickformat=\'.1%\')\n    training = go.Figure()\n    for seed, frame in history.groupby(\'seed\', sort=False):\n        training.add_trace(go.Scatter(x=frame.step+1, y=frame.mean_reward, name=f\'Seed {seed} · mean reward\'))\n    training.update_layout(title=\'Training reward by seed\', xaxis_title=\'Policy update step\', yaxis_title=\'Mean train score\')\n    for fig in [equity, drawdown, training]:\n        fig.update_layout(template=\'plotly_white\', height=380, margin=dict(l=50,r=20,t=55,b=45),\n                          legend=dict(orientation=\'h\', y=-0.2))\n    return equity, drawdown, training\n\n\ndef render_results(directory):\n    root = Path(directory); figures = root/\'figures\'; figures.mkdir(exist_ok=True)\n    previews = figures/\'previews\'; previews.mkdir(exist_ok=True)\n    # Keep raster-only QA previews separate from the nine PNG/SVG deliverables.\n    for previous in figures.glob(\'*_grayscale.png\'):\n        previous.replace(previews/previous.name)\n    best, series, baselines, history, candidates = load_results(root)\n    candles = pd.read_csv(root/\'data/candles.csv\')\n    features = pd.read_csv(root/\'features.csv\')\n    warmup = int(np.searchsorted(candles.time, int(pd.Timestamp(best[\'splits\'][\'train\'][\'signal_start_utc\']).timestamp())))\n    setup_style(journal=\'general\', lang=\'en\', use_sciplots=False)\n    plt.rcParams.update({\'font.size\':10, \'axes.titlesize\':12, \'legend.fontsize\':9})\n    dates = pd.to_datetime(candles.time, unit=\'s\', utc=True)\n    test_dates = pd.to_datetime(series.exit_time, unit=\'s\', utc=True)\n    contracts, pngs = [], []\n\n    def chart(name, claim, draw, xlabel, ylabel):\n        fig, ax = plt.subplots(figsize=(8,4), layout=\'constrained\')\n        draw(ax)\n        ax.set(xlabel=xlabel, ylabel=ylabel)\n        ax.set_title(claim, loc=\'left\', pad=12)\n        ax.spines[[\'top\',\'right\']].set_visible(False)\n        if xlabel == \'UTC\':\n            import matplotlib.dates as mdates\n            ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=5))\n            ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))\n        issues = audit_layout(fig)\n        if issues: raise RuntimeError(f\'Figure layout audit: {name}: {issues}\')\n        export_figure(fig, str(figures/name), formats=[\'svg\',\'png\'], size_inches=(8,4), dpi=300,\n                      grayscale_preview=False, tight=False)\n        from PIL import Image\n        with Image.open(figures/f\'{name}.png\') as im:\n            im.convert(\'L\').save(previews/f\'{name}_grayscale.png\', dpi=(300,300))\n        plt.close(fig)\n        pngs.append(str(figures/f\'{name}.png\'))\n        contracts.append({\'file\':name, \'question\':\'q1\', \'claim\':claim, \'size_inches\':[8,4],\n                          \'backend\':\'matplotlib\', \'dpi\':300, \'layout\':\'single quantitative panel\',\n                          \'x\':xlabel, \'y\':ylabel, \'uncertainty\':\'descriptive observed series; no confidence intervals\'})\n\n    chart(\'raw_q1_price\', \'BTC perpetual price and chronological splits\', lambda ax: (\n        ax.plot(dates, candles.close, color=BLUE, linewidth=0.9),\n        [ax.axvline(pd.Timestamp(best[\'splits\'][s][\'signal_start_utc\']), color=ORANGE, linestyle=\'--\',\n                    label=s.title()) for s in [\'validation\',\'test\']], ax.legend()), \'UTC\', \'Close (USDT)\')\n    chart(\'raw_q1_volume\', \'Distribution of hourly USDT turnover\',\n          lambda ax: ax.hist(np.log10(candles.volume.clip(lower=1)), bins=45, color=BLUE),\n          \'log10(hourly turnover / USDT)\', \'Hours\')\n    names = [\'RET\',\'MOM24\',\'VOL_SHOCK\',\'DEV24\',\'PRESSURE\',\'VOLATILITY24\']\n    chart(\'raw_q1_features\', \'Causal normalized feature distributions\',\n          lambda ax: ax.boxplot([features[n].iloc[warmup:] for n in names], tick_labels=names,\n                                showfliers=False), \'Feature (warmup excluded)\', \'Normalized value\')\n    chart(\'process_q1_reward\', \'Mean training reward by independent seed\', lambda ax: (\n        [ax.plot(f.step+1, f.mean_reward, label=str(s)) for s,f in history.groupby(\'seed\',sort=False)],\n        ax.legend(title=\'Seed\')), \'Policy update step\', \'Mean train score\')\n    chart(\'process_q1_archive\', \'Best observed training score in search archive\',\n          lambda ax: ax.plot(np.arange(len(history))+1, history.best_train_score, color=BLUE),\n          \'Total policy update step\', \'Best train score\')\n    chart(\'process_q1_candidates\', \'Training shortlist compared on validation only\', lambda ax: (\n        ax.scatter(candidates.train_score, candidates.validation_score, color=BLUE),\n        ax.scatter(candidates.iloc[0].train_score, candidates.iloc[0].validation_score,\n                   marker=\'*\', s=140, color=ORANGE, label=\'Selected\'), ax.legend()),\n          \'Train score\', \'Validation score\')\n    chart(\'result_q1_equity\', \'Test equity after fees, slippage and funding\', lambda ax: (\n        ax.plot(test_dates, series.equity, color=BLUE, label=\'Strategy\'),\n        ax.plot(test_dates, baselines.long_1x, color=ORANGE, linestyle=\'--\',label=\'Long 1x\'),\n        ax.plot(test_dates, baselines.short_1x, color=GREEN, linestyle=\':\',label=\'Short 1x\'),\n        ax.axhline(1, color=\'#777777\', linewidth=0.7), ax.legend()), \'UTC\', \'Equity (initial = 1)\')\n    peaks = np.maximum.accumulate(np.r_[1.0, series.equity])[1:]\n    chart(\'result_q1_drawdown\', \'Test drawdown includes the initial equity peak\',\n          lambda ax: ax.fill_between(test_dates, 0, 100*(1-series.equity/peaks), color=ORANGE, alpha=0.6),\n          \'UTC\', \'Drawdown (%)\')\n    chart(\'result_q1_positions\', \'Test exposure: long, cash or short\', lambda ax: (\n        ax.step(pd.to_datetime(series.entry_time,unit=\'s\',utc=True), series.position,where=\'post\',color=BLUE),\n        ax.set_yticks([-1,0,1], [\'Short\',\'Cash\',\'Long\'])), \'UTC\', \'Direction (target 1x)\')\n    (figures/\'contracts.json\').write_text(json.dumps(contracts,indent=2),encoding=\'utf-8\')\n    plots = interactive_figures(root)\n    report = \'<!doctype html><html><meta charset="utf-8"><title>AlphaGPT BTC report</title><body style="max-width:1100px;margin:auto;font-family:Arial">\'\n    report += \'<h1>AlphaGPT BTC · research results</h1><pre>\'+html.escape(best[\'formula\'])+\'</pre>\'\n    report += \'<p>Selection uses validation only. Test return: \'+f\'{best["test"]["total_return"]:.2%}\'+\'. Best observed candidate does not guarantee profitability.</p>\'\n    for i, fig in enumerate(plots): report += fig.to_html(full_html=False, include_plotlyjs=True if i==0 else False)\n    report += \'\'.join(f\'<img src="figures/{Path(p).name}" alt="{Path(p).stem}" style="width:100%">\' for p in pngs)\n    (root/\'report.html\').write_text(report+\'</body></html>\',encoding=\'utf-8\')\n    return pngs\n', 'btc_mining/core.py': '"""Gate BTC perpetual formula mining using the repository\'s AlphaGPT generator.\n\nNo keys, database or live trading. All times are UTC; end is exclusive.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict, dataclass, field\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport torch\nfrom torch import nn\nfrom torch.distributions import Categorical\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom model_core.alphagpt import AlphaGPT, MTPHead\n\nFEATURES = [\'RET\', \'MOM24\', \'VOL_SHOCK\', \'DEV24\', \'PRESSURE\', \'VOLATILITY24\']\nOPS = [(\'ADD\', 2), (\'SUB\', 2), (\'MUL\', 2), (\'DIV\', 2), (\'NEG\', 1),\n       (\'ABS\', 1), (\'SIGN\', 1), (\'GATE\', 3), (\'JUMP\', 1),\n       (\'DECAY\', 1), (\'DELAY1\', 1), (\'MAX3\', 1)]\nTOKENS = FEATURES + [x[0] for x in OPS] + [\'EOS\']\nEOS = len(TOKENS) - 1\nSECONDS = {\'1h\': 3600, \'4h\': 14400}\n\n\ndef recent_date(days=0):\n    return str((pd.Timestamp.now(tz=\'UTC\').normalize() - pd.Timedelta(days=days)).date())\n\n\n@dataclass\nclass Config:\n    contract: str = \'BTC_USDT\'\n    interval: str = \'1h\'\n    start: str = field(default_factory=lambda: recent_date(170))\n    end: str = field(default_factory=recent_date)\n    steps: int = 200\n    batch_size: int = 128\n    max_len: int = 12\n    seeds: tuple = (42, 43, 44)\n    learning_rate: float = 0.0003\n    fee_bps: float = 5.0\n    slippage_bps: float = 2.0\n    signal_threshold: float = 0.2\n    norm_window: int = 168\n    train_fraction: float = 0.6\n    validation_fraction: float = 0.2\n    top_k: int = 30\n    min_entries: int = 10\n    drawdown_penalty: float = 2.0\n    complexity_penalty: float = 0.005\n    api_base: str = \'https://api.gateio.ws/api/v4\'\n    charts: bool = True\n\n    def validate(self):\n        integer_fields = [\'steps\', \'batch_size\', \'max_len\', \'norm_window\', \'top_k\', \'min_entries\']\n        if any(isinstance(getattr(self, k), bool) or not isinstance(getattr(self, k), int) for k in integer_fields):\n            raise ValueError(\'Training counts/window sizes must be integers\')\n        if any(not isinstance(s, int) or not 0 <= s < 2**32 for s in self.seeds):\n            raise ValueError(\'Seeds must be integers in [0, 2**32)\')\n        if any(not np.isfinite(getattr(self,k)) for k in [\'learning_rate\',\'fee_bps\',\'slippage_bps\',\n            \'signal_threshold\',\'train_fraction\',\'validation_fraction\',\'drawdown_penalty\',\'complexity_penalty\']):\n            raise ValueError(\'All numeric parameters must be finite\')\n        if self.learning_rate <= 0 or min(self.drawdown_penalty, self.complexity_penalty) < 0:\n            raise ValueError(\'Learning rate must be positive; penalties nonnegative\')\n        if self.interval not in SECONDS or not 2 <= self.max_len <= 12:\n            raise ValueError(\'interval must be 1h/4h; max_len must be 2..12\')\n        if self.steps < 1 or self.batch_size < 2 or not self.seeds:\n            raise ValueError(\'steps >= 1, batch_size >= 2 and seeds required\')\n        if self.norm_window < 24 or self.top_k < 1 or self.min_entries < 1:\n            raise ValueError(\'norm_window >= 24, top_k >= 1, min_entries >= 1\')\n        if not (0 < self.train_fraction < 1 and 0 < self.validation_fraction < 1\n                and self.train_fraction + self.validation_fraction < 1):\n            raise ValueError(\'Invalid chronological split fractions\')\n        if not 0 <= self.signal_threshold < 1 or min(self.fee_bps, self.slippage_bps) < 0:\n            raise ValueError(\'Invalid threshold or trading costs\')\n        start, end = utc_seconds(self.start), utc_seconds(self.end)\n        dt = SECONDS[self.interval]\n        if start % dt or end % dt or start >= end:\n            raise ValueError(\'Dates must be aligned UTC candle boundaries; start < end\')\n        if end > int(time.time()) // dt * dt:\n            raise ValueError(\'end includes an unfinished/future candle\')\n\n\ndef utc_seconds(value):\n    ts = pd.Timestamp(value)\n    ts = ts.tz_localize(\'UTC\') if ts.tzinfo is None else ts.tz_convert(\'UTC\')\n    return int(ts.timestamp())\n\n\ndef get_json(session, url, params, retries=5):\n    for attempt in range(retries):\n        try:\n            resp = session.get(url, params=params, timeout=(10, 45))\n            if resp.status_code == 429 or resp.status_code >= 500:\n                raise requests.RequestException(f\'HTTP {resp.status_code}\')\n            resp.raise_for_status()\n            data = resp.json()\n            if not isinstance(data, list):\n                raise ValueError(f\'Expected Gate list response, got {str(data)[:200]}\')\n            return data\n        except requests.RequestException:\n            if attempt == retries - 1:\n                raise\n            time.sleep(min(2 ** attempt, 16))\n    raise RuntimeError(\'Unreachable\')\n\n\ndef validate_candles(df, cfg):\n    df = df.copy()\n    required = [\'time\', \'open\', \'high\', \'low\', \'close\', \'volume\']\n    if not set(required).issubset(df.columns):\n        raise ValueError(f\'CSV must contain {required}\')\n    for name in required:\n        df[name] = pd.to_numeric(df[name], errors=\'raise\')\n    if not np.isfinite(df[required].to_numpy()).all():\n        raise ValueError(\'NaN/inf in candles\')\n    if not (df.time == df.time.astype(\'int64\')).all():\n        raise ValueError(\'Candle times must be integer Unix seconds\')\n    df[\'time\'] = df.time.astype(\'int64\')\n    df = df.drop_duplicates(\'time\').sort_values(\'time\').reset_index(drop=True)\n    start, end, dt = utc_seconds(cfg.start), utc_seconds(cfg.end), SECONDS[cfg.interval]\n    df = df[(df.time >= start) & (df.time < end)].reset_index(drop=True)\n    expected = np.arange(start, end, dt, dtype=np.int64)\n    if not np.array_equal(df.time.to_numpy(), expected):\n        missing = np.setdiff1d(expected, df.time.to_numpy())\n        raise ValueError(f\'Incomplete/misaligned candles: {len(missing)} missing; \'\n                         f\'first missing: {missing[:5].tolist()}. No silent forward filling.\')\n    p = df[[\'open\', \'high\', \'low\', \'close\']]\n    if (p <= 0).any().any() or (df.volume < 0).any():\n        raise ValueError(\'Nonpositive price or negative quote volume\')\n    if ((df.high < p[[\'open\', \'close\', \'low\']].max(axis=1)) |\n            (df.low > p[[\'open\', \'close\', \'high\']].min(axis=1))).any():\n        raise ValueError(\'Invalid OHLC ordering\')\n    return df\n\n\ndef fetch_gate(cfg, data_dir):\n    """Bounded nonoverlapping windows, strict complete candle grid, cached CSVs."""\n    cfg.validate()\n    data_dir = Path(data_dir)\n    data_dir.mkdir(parents=True, exist_ok=True)\n    key = hashlib.sha256(json.dumps({k: getattr(cfg, k) for k in\n        [\'contract\', \'interval\', \'start\', \'end\', \'api_base\']}, sort_keys=True).encode()).hexdigest()[:12]\n    candle_path, funding_path = data_dir / f\'candles_{key}.csv\', data_dir / f\'funding_{key}.csv\'\n    start, end, dt = utc_seconds(cfg.start), utc_seconds(cfg.end), SECONDS[cfg.interval]\n    if not funding_path.exists() and start < int(time.time()) - 179 * 86400:\n        raise ValueError(\'Gate funding API currently rejects history older than 180 days. \'\n                         \'Use the default recent 170 days, or supply historical candle AND funding CSVs.\')\n    with requests.Session() as session:\n        if candle_path.exists():\n            candles = validate_candles(pd.read_csv(candle_path), cfg)\n        else:\n            rows = []\n            for cursor in range(start, end, 1000 * dt):\n                stop = min(cursor + 1000 * dt, end)\n                batch = get_json(session, cfg.api_base + \'/futures/usdt/candlesticks\',\n                    {\'contract\': cfg.contract, \'interval\': cfg.interval, \'from\': cursor, \'to\': stop - 1})\n                for x in batch:\n                    rows.append({\'time\': x[\'t\'], \'open\': x[\'o\'], \'high\': x[\'h\'],\n                        \'low\': x[\'l\'], \'close\': x[\'c\'], \'volume\': x[\'sum\']})\n                print(f\'Candles: {pd.Timestamp(stop, unit="s", tz="UTC")} ({len(rows)} rows)\', flush=True)\n                time.sleep(0.1)\n            candles = validate_candles(pd.DataFrame(rows), cfg)\n            candles.to_csv(candle_path, index=False)\n        if funding_path.exists():\n            funding = pd.read_csv(funding_path)\n        else:\n            rows = []\n            # Seven-day windows contain <=168 records even at hourly settlement.\n            for cursor in range(start, end, 7 * 86400):\n                stop = min(cursor + 7 * 86400, end)\n                batch = get_json(session, cfg.api_base + \'/futures/usdt/funding_rate\',\n                    {\'contract\': cfg.contract, \'from\': cursor, \'to\': stop - 1, \'limit\': 1000})\n                if not batch:\n                    raise ValueError(f\'No funding history in UTC day {cursor}; refusing zero-cost fallback\')\n                rows.extend({\'time\': x[\'t\'], \'rate\': x[\'r\']} for x in batch)\n                print(f\'Funding: {pd.Timestamp(stop, unit="s", tz="UTC")} ({len(rows)} rows)\', flush=True)\n                time.sleep(0.05)\n            funding = pd.DataFrame(rows).drop_duplicates(\'time\').sort_values(\'time\')\n            funding.to_csv(funding_path, index=False)\n    funding = validate_funding(funding, cfg)\n    return candles, funding, [candle_path, funding_path]\n\n\ndef validate_funding(funding, cfg):\n    funding = funding.copy()\n    for col in [\'time\', \'rate\']:\n        funding[col] = pd.to_numeric(funding[col], errors=\'raise\')\n    if funding.empty or not np.isfinite(funding[[\'time\', \'rate\']].to_numpy()).all():\n        raise ValueError(\'Empty/nonfinite funding history\')\n    if not (funding.time == funding.time.astype(\'int64\')).all():\n        raise ValueError(\'Funding timestamps must be integer Unix seconds\')\n    funding[\'time\'] = funding.time.astype(\'int64\')\n    funding = funding.drop_duplicates(\'time\').sort_values(\'time\').reset_index(drop=True)\n    start, end = utc_seconds(cfg.start), utc_seconds(cfg.end)\n    funding = funding[(funding.time >= start) & (funding.time < end)].reset_index(drop=True)\n    stamps = np.r_[start, funding.time.to_numpy(), end]\n    if funding.empty or np.diff(stamps).max() > 12 * 3600:\n        raise ValueError(\'Funding coverage gap >12h; inspect source instead of assuming zero\')\n    return funding\n\n\ndef make_features(df, window=168):\n    """All features use current/past CLOSED bars; no fitted full-series statistics."""\n    c, v = df.close, df.volume\n    ret = np.log(c / c.shift(1))\n    raw = pd.DataFrame({\n        \'RET\': ret, \'MOM24\': np.log(c / c.shift(24)),\n        \'VOL_SHOCK\': np.log((v + 1) / (v.rolling(24).mean() + 1)),\n        \'DEV24\': c / c.rolling(24).mean() - 1,\n        \'PRESSURE\': (c - df.open) / (df.high - df.low).clip(lower=1e-9),\n        \'VOLATILITY24\': ret.rolling(24).std(ddof=0),\n    })\n    med = raw.rolling(window, min_periods=window).median()\n    # Exact rolling MAD over the same window (not a second rolling approximation).\n    mad = raw.rolling(window, min_periods=window).apply(\n        lambda x: np.median(np.abs(x - np.median(x))), raw=True)\n    feats = ((raw - med) / (mad + 1e-6)).clip(-5, 5)\n    good = np.isfinite(feats.to_numpy()).all(axis=1)\n    if not good.any():\n        raise ValueError(\'Too few bars for causal feature warmup\')\n    warmup = int(np.flatnonzero(good)[0])\n    if not good[warmup:].all():\n        raise ValueError(\'Unexpected invalid features after warmup\')\n    return feats.fillna(0).to_numpy(dtype=np.float64).T, warmup\n\n\ndef delay(x, n):\n    out = np.zeros_like(x)\n    if n < len(x):\n        out[n:] = x[:-n]\n    return out\n\n\ndef execute(tokens, features):\n    stack = []\n    with np.errstate(all=\'ignore\'):\n        for token in tokens:\n            if token == EOS:\n                break\n            if 0 <= token < len(FEATURES):\n                stack.append(features[token])\n                continue\n            if not len(FEATURES) <= token < EOS:\n                raise ValueError(\'Unknown token\')\n            name, arity = OPS[token - len(FEATURES)]\n            if len(stack) < arity:\n                raise ValueError(\'Formula stack underflow\')\n            args = stack[-arity:]\n            del stack[-arity:]\n            x = args[0]\n            if name == \'ADD\': out = x + args[1]\n            elif name == \'SUB\': out = x - args[1]\n            elif name == \'MUL\': out = x * args[1]\n            elif name == \'DIV\':\n                y = args[1]\n                out = x / np.where(np.abs(y) < 1e-6, np.where(y < 0, -1e-6, 1e-6), y)\n            elif name == \'NEG\': out = -x\n            elif name == \'ABS\': out = np.abs(x)\n            elif name == \'SIGN\': out = np.sign(x)\n            elif name == \'GATE\': out = np.where(x > 0, args[1], args[2])\n            elif name == \'JUMP\':\n                s = pd.Series(x)\n                z = (s - s.rolling(20).mean()) / (s.rolling(20).std(ddof=0) + 1e-6)\n                out = np.maximum(z.fillna(0).to_numpy() - 3, 0)\n            elif name == \'DECAY\': out = x + 0.8 * delay(x, 1) + 0.6 * delay(x, 2)\n            elif name == \'DELAY1\': out = delay(x, 1)\n            else: out = np.maximum(x, np.maximum(delay(x, 1), delay(x, 2)))\n            stack.append(np.clip(np.nan_to_num(out, nan=0, posinf=1e6, neginf=-1e6), -1e6, 1e6))\n    if len(stack) != 1:\n        raise ValueError(\'Formula must leave exactly one result\')\n    return stack[0]\n\n\ndef decode(tokens):\n    stack = []\n    for t in tokens:\n        if t == EOS: break\n        if t < len(FEATURES): stack.append(FEATURES[t])\n        else:\n            name, arity = OPS[t - len(FEATURES)]\n            if len(stack) < arity: raise ValueError(\'Invalid formula\')\n            args = stack[-arity:]\n            del stack[-arity:]\n            stack.append(f\'{name}({", ".join(args)})\')\n    if len(stack) != 1: raise ValueError(\'Invalid formula\')\n    return stack[0]\n\n\nclass Backtester:\n    def __init__(self, df, funding, cfg, warmup):\n        self.cfg = cfg\n        self.df = df\n        self.periods_per_year = 365.25 * 86400 / SECONDS[cfg.interval]\n        self.ret = np.zeros(len(df))\n        self.ret[:-2] = df.open.to_numpy()[2:] / df.open.to_numpy()[1:-1] - 1\n        self.funding = np.zeros(len(df))\n        times = df.time.to_numpy()\n        # signal[t] owns (open[t+1], open[t+2]], including settlement at exit.\n        for stamp, rate in funding[[\'time\', \'rate\']].itertuples(index=False, name=None):\n            i = int(np.searchsorted(times, stamp, side=\'left\')) - 2\n            if 0 <= i < len(df) - 2:\n                self.funding[i] += rate\n        a, b = int(len(df) * cfg.train_fraction), int(len(df) * (cfg.train_fraction + cfg.validation_fraction))\n        # Purge target horizons crossing each boundary; restart flat for each segment.\n        self.splits = {\'train\': (warmup, a - 2), \'validation\': (a, b - 2), \'test\': (b, len(df) - 2)}\n        if any(stop - start < 100 for start, stop in self.splits.values()):\n            raise ValueError(\'Each split needs >=100 tradable bars after warmup/purge\')\n\n    def evaluate(self, signal, split, length=0, positions=False):\n        start, stop = self.splits[split]\n        s = np.tanh(signal[start:stop])\n        p = np.where(s > self.cfg.signal_threshold, 1.0,\n                     np.where(s < -self.cfg.signal_threshold, -1.0, 0.0))\n        return self.evaluate_positions(p, start, stop, length, positions)\n\n    def evaluate_positions(self, p, start, stop, length=0, include_series=False):\n        prev = np.r_[0.0, p[:-1]]\n        # Target exposure is reset to +/-1 equity at each open. Include resizing\n        # of an unchanged direction after price/funding changes, as well as flips.\n        r, f = self.ret[start:stop], self.funding[start:stop]\n        cost_rate = (self.cfg.fee_bps + self.cfg.slippage_bps) / 1e4\n        turnover = np.abs(p - prev)\n        entries = int(np.count_nonzero((p != 0) & (p != prev)))\n        costs = turnover * cost_rate\n        net = p * r - p * f - costs\n        # Charge approximate maintenance rebalancing on same-direction bars.\n        same = (p[1:] != 0) & (p[1:] == p[:-1])\n        resize = np.abs(p[:-1] * (1 + r[:-1]) / (1 + net[:-1]) - p[1:])\n        costs[1:] += np.where(same, resize * cost_rate, 0)\n        net = p * r - p * f - costs\n        net[-1] -= abs(p[-1]) * cost_rate  # liquidate at segment end\n        if not np.isfinite(net).all() or (net <= -1).any():\n            raise ValueError(\'Nonfinite return or bankruptcy in simplified 1x model\')\n        equity = np.cumprod(1 + net)\n        peaks = np.maximum.accumulate(np.r_[1.0, equity])[1:]\n        dd = float(np.max(1 - equity / peaks))\n        sd = float(net.std(ddof=0))\n        sharpe = float(net.mean() / sd * np.sqrt(self.periods_per_year)) if sd > 1e-12 else 0.0\n        reward = float(np.clip(sharpe, -10, 10) - self.cfg.drawdown_penalty * dd\n                       - self.cfg.complexity_penalty * length)\n        eligible = entries >= self.cfg.min_entries\n        result = {\'score\': reward if eligible else -20.0, \'eligible\': bool(eligible),\n            \'total_return\': float(equity[-1] - 1), \'sharpe\': sharpe,\n            \'max_drawdown\': dd, \'entries\': entries,\n            \'long_fraction\': float((p > 0).mean()), \'short_fraction\': float((p < 0).mean()),\n            \'bars\': len(p)}\n        if include_series:\n            result[\'series\'] = pd.DataFrame({\'signal_time\': self.df.time.iloc[start:stop].to_numpy(),\n                \'entry_time\': self.df.time.iloc[start + 1:stop + 1].to_numpy(),\n                \'exit_time\': self.df.time.iloc[start + 2:stop + 2].to_numpy(),\n                \'position\': p, \'market_return\': r, \'funding_rate\': f,\n                \'net_return\': net, \'equity\': equity})\n        return result\n\n\ndef make_generator(device):\n    # Keep the actual repository\'s looped AlphaGPT architecture; replace token heads\n    # for BTC\'s 6 features + 12 operators + explicit EOS. LoRD remains disabled.\n    model = AlphaGPT().to(device)\n    model.token_emb = nn.Embedding(len(TOKENS), model.d_model).to(device)\n    model.mtp_head = MTPHead(model.d_model, len(TOKENS), num_tasks=3).to(device)\n    model.vocab = TOKENS\n    model.vocab_size = len(TOKENS)\n    return model\n\n\ndef sample_formulas(model, batch_size, max_len, device):\n    inp = torch.zeros((batch_size, 1), dtype=torch.long, device=device)\n    depth = torch.zeros(batch_size, dtype=torch.long, device=device)\n    done = torch.zeros(batch_size, dtype=torch.bool, device=device)\n    arities = torch.tensor([0] * len(FEATURES) + [a for _, a in OPS] + [0], device=device)\n    sums = torch.zeros(batch_size, device=device)\n    seqs = []\n    for step in range(max_len):\n        logits, _, _ = model(inp)\n        after = depth[:, None] + torch.where(arities[None, :] == 0, 1, 1 - arities[None, :])\n        valid = depth[:, None] >= arities[None, :]\n        remaining = max_len - step - 1\n        valid &= (after >= 1) & (after <= 1 + 2 * remaining)\n        valid[:, EOS] = depth == 1\n        valid[done] = False\n        valid[done, EOS] = True\n        dist = Categorical(logits=logits.masked_fill(~valid, -torch.inf))\n        action = dist.sample()\n        sums = sums + dist.log_prob(action) * (~done)\n        seqs.append(action)\n        new_done = done | (action == EOS)\n        depth = torch.where(new_done, depth, depth + torch.where(action < len(FEATURES), 1, 1 - arities[action]))\n        done = new_done\n        inp = torch.cat([inp, action[:, None]], dim=1)\n    tokens = torch.stack(seqs, dim=1).detach().cpu().tolist()\n    tokens = [tuple(s[:s.index(EOS)] if EOS in s else s) for s in tokens]\n    return tokens, sums\n\n\ndef search(features, bt, cfg, progress=None):\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if device.type == \'cpu\': torch.set_num_threads(min(4, torch.get_num_threads()))\n    cache, history = {}, []\n    # Seed the archive with interpretable single-feature baselines; learned\n    # candidates must compete against them under the same costs and selection.\n    for i in range(len(FEATURES)):\n        for formula in [(i,), (i, len(FEATURES) + 4)]:\n            signal = execute(formula, features)\n            cache[formula] = bt.evaluate(signal, \'train\', len(formula))\n    for seed in cfg.seeds:\n        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n        model = make_generator(device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)\n        for step in range(cfg.steps):\n            seqs, logp = sample_formulas(model, cfg.batch_size, cfg.max_len, device)\n            values = []\n            for formula in seqs:\n                if formula not in cache:\n                    signal = execute(formula, features)\n                    cache[formula] = bt.evaluate(signal, \'train\', len(formula))\n                values.append(cache[formula][\'score\'])\n            rewards = torch.tensor(values, dtype=torch.float32, device=device)\n            adv = (rewards - rewards.mean()) / rewards.std(unbiased=False).clamp(min=1e-6)\n            loss = -(logp * adv.detach()).mean()\n            optimizer.zero_grad(); loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            history.append({\'seed\': seed, \'step\': step, \'mean_reward\': float(rewards.mean()),\n                            \'best_train_score\': max(x[\'score\'] for x in cache.values()),\n                            \'unique_formulas\': len(cache)})\n            if progress:\n                progress((len(history) / (cfg.steps * len(cfg.seeds))),\n                         f\'Seed {seed} · step {step+1}/{cfg.steps} · {len(cache)} formulas\')\n            if step % 10 == 0 or step == cfg.steps - 1:\n                print(f\'seed={seed} step={step+1}/{cfg.steps} formulas={len(cache)} \'\n                      f\'best_train={history[-1]["best_train_score"]:.3f}\', flush=True)\n        del model, optimizer\n        if device.type == \'cuda\': torch.cuda.empty_cache()\n    ranked = sorted((f for f in cache if cache[f][\'eligible\']),\n                    key=lambda f: (-cache[f][\'score\'], len(f), f))[:cfg.top_k]\n    if not ranked: raise RuntimeError(\'No eligible training formula; inspect data/search configuration\')\n    candidates = []\n    for formula in ranked:\n        val = bt.evaluate(execute(formula, features), \'validation\', len(formula))\n        if val[\'eligible\']:\n            candidates.append({\'tokens\': list(formula), \'formula\': decode(formula),\n                               \'train\': cache[formula], \'validation\': val})\n    if not candidates: raise RuntimeError(\'No eligible validation formula; no best formula exported\')\n    candidates.sort(key=lambda x: (-x[\'validation\'][\'score\'], len(x[\'tokens\']), x[\'tokens\']))\n    return candidates[0], candidates, history, device.type\n\n\ndef run(cfg, output_dir=\'btc_results\', candles_csv=None, funding_csv=None, progress=None):\n    cfg.validate()\n    started = time.time()\n    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)\n    # Each result directory is a frozen run; never mix old success files with a failed retry.\n    if (out / \'best_btc_formula.json\').exists():\n        raise ValueError(\'Output already contains a completed run. Choose a new --output directory.\')\n    (out / \'config.json\').write_text(json.dumps(asdict(cfg), indent=2), encoding=\'utf-8\')\n    if progress: progress(0, \'Fetching and validating Gate candles / funding\')\n    if candles_csv or funding_csv:\n        if not candles_csv or not funding_csv: raise ValueError(\'Both candle and funding CSV paths required\')\n        df = validate_candles(pd.read_csv(candles_csv), cfg)\n        funding = validate_funding(pd.read_csv(funding_csv), cfg)\n        inputs = [Path(candles_csv), Path(funding_csv)]\n    else:\n        df, funding, inputs = fetch_gate(cfg, out / \'data\')\n    features, warmup = make_features(df, cfg.norm_window)\n    warmup += cfg.max_len * 20  # conservative nested temporal-operator warmup\n    bt = Backtester(df, funding, cfg, warmup)\n    best, candidates, history, device = search(features, bt, cfg, progress=progress)\n    # The test split is accessed only after selection has finished.\n    tested = bt.evaluate(execute(best[\'tokens\'], features), \'test\', len(best[\'tokens\']), positions=True)\n    series = tested.pop(\'series\')\n    best[\'test\'] = tested\n    baselines = {}\n    baseline_series = {}\n    a, b = bt.splits[\'test\']\n    for name, p in [(\'long_1x\', np.ones(b-a)), (\'short_1x\', -np.ones(b-a)), (\'cash\', np.zeros(b-a))]:\n        evaluated = bt.evaluate_positions(p, a, b, include_series=True)\n        baseline_series[name] = evaluated.pop(\'series\')[\'equity\'].to_numpy()\n        baselines[name] = evaluated\n    best.update({\'schema_version\': 1, \'market\': \'Gate USDT perpetual\',\n        \'contract\': cfg.contract, \'interval\': cfg.interval, \'selection\': \'train top-k, then validation score; test untouched\',\n        \'config\': asdict(cfg), \'vocabulary\': TOKENS,\n        \'feature_contract\': \'causal OHLCV, rolling median / exact rolling MAD, clipped [-5,5]\',\n        \'operator_contract\': \'BTC VM v1; JUMP uses trailing 20-bar zscore; DIV sign-preserving epsilon; outputs clipped +/-1e6\',\n        \'signal_rule\': f\'tanh(formula) > {cfg.signal_threshold}: long; < -{cfg.signal_threshold}: short; otherwise cash\',\n        \'execution_rule\': \'closed bar t -> open[t+1] entry -> open[t+2] mark/exit; 1x target notional; funding at exit belongs to incumbent position\',\n        \'limitations\': [\'hourly OHLC fills and approximate notional resizing, no order book/mark-price liquidation model\',\n                        \'validation selection is susceptible to multiple testing; test is descriptive, not proof of profitability\',\n                        \'BTC vocabulary is incompatible with the original Solana strategy runner\'],\n        \'test_baselines\': baselines,\n        \'quality_flag\': \'positive_test_return\' if tested[\'total_return\'] > 0 else \'test_nonpositive_do_not_treat_as_profitable\',\n        \'splits\': {k: {\'signal_start_utc\': str(pd.Timestamp(df.time.iloc[a], unit=\'s\', tz=\'UTC\')),\n                      \'last_signal_utc\': str(pd.Timestamp(df.time.iloc[b-1], unit=\'s\', tz=\'UTC\')),\n                      \'bars\': b-a} for k, (a,b) in bt.splits.items()}})\n    (out / \'best_btc_formula.json\').write_text(json.dumps(best, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n    summary = \'\\n\'.join([f\'Formula: {best["formula"]}\', f\'Tokens: {best["tokens"]}\', best[\'signal_rule\'],\n        best[\'execution_rule\'], f\'Validation: {best["validation"]}\', f\'Test: {best["test"]}\',\n        f\'Quality: {best["quality_flag"]}\', \'Best means best observed validation candidate, not a global optimum.\'])\n    (out / \'best_btc_formula.txt\').write_text(summary, encoding=\'utf-8\')\n    pd.DataFrame([{\'formula\': x[\'formula\'], \'tokens\': json.dumps(x[\'tokens\']),\n                  **{f\'train_{k}\': v for k,v in x[\'train\'].items()},\n                  **{f\'validation_{k}\': v for k,v in x[\'validation\'].items()}} for x in candidates]).to_csv(out / \'candidates.csv\', index=False)\n    pd.DataFrame(history).to_csv(out / \'training_history.csv\', index=False)\n    series.to_csv(out / \'test_equity.csv\', index=False)\n    pd.DataFrame({\'exit_time\': series.exit_time, **baseline_series}).to_csv(out / \'test_baselines.csv\', index=False)\n    pd.DataFrame(features.T, columns=FEATURES).assign(time=df.time.to_numpy()).to_csv(out / \'features.csv\', index=False)\n    # Snapshot inputs even when supplied by upload, so downloaded runs are reproducible.\n    data = out / \'data\'; data.mkdir(exist_ok=True)\n    df.to_csv(data / \'candles.csv\', index=False)\n    funding.to_csv(data / \'funding.csv\', index=False)\n    inputs = [data / \'candles.csv\', data / \'funding.csv\']\n    manifest = {\'config\': asdict(cfg), \'device\': device, \'runtime_seconds\': time.time()-started,\n        \'python\': sys.version, \'versions\': {p: importlib.metadata.version(p) for p in [\'torch\', \'numpy\', \'pandas\', \'requests\']},\n        \'input_sha256\': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs},\n        \'code_sha256\': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in\n            [Path(__file__)] + [Path(__file__).resolve().parents[1] / \'model_core\' / n\n                for n in [\'alphagpt.py\',\'config.py\',\'vocab.py\',\'ops.py\']]}}\n    (out / \'reproduction.json\').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    if cfg.charts:\n        if progress: progress(1, \'Exporting figures and interactive report\')\n        from .charts import render_results\n        render_results(out)\n        manifest[\'versions\'].update({p:importlib.metadata.version(p) for p in [\'matplotlib\',\'plotly\',\'pillow\']})\n        for chart_source in [Path(__file__).with_name(\'charts.py\')] + list(Path(__file__).with_name(\'figure_tools\').glob(\'*.py\')):\n            manifest[\'code_sha256\'][str(chart_source)] = hashlib.sha256(chart_source.read_bytes()).hexdigest()\n        manifest[\'runtime_seconds\'] = time.time()-started\n        (out/\'reproduction.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(summary, flush=True)\n    return best\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', help=\'JSON Config dictionary\')\n    parser.add_argument(\'--output\', default=\'btc_results\')\n    parser.add_argument(\'--candles-csv\'); parser.add_argument(\'--funding-csv\')\n    parser.add_argument(\'--steps\', \'--epochs\', type=int, help=\'Policy-gradient steps per seed (not data epochs)\')\n    parser.add_argument(\'--batch-size\', type=int)\n    parser.add_argument(\'--seeds\', help=\'Comma-separated integer seeds, e.g. 42,43,44\')\n    parser.add_argument(\'--start\'); parser.add_argument(\'--end\')\n    parser.add_argument(\'--interval\', choices=SECONDS)\n    parser.add_argument(\'--learning-rate\', type=float)\n    parser.add_argument(\'--max-len\', type=int)\n    parser.add_argument(\'--top-k\', type=int)\n    parser.add_argument(\'--min-entries\', type=int)\n    parser.add_argument(\'--norm-window\', type=int)\n    parser.add_argument(\'--fee-bps\', type=float); parser.add_argument(\'--slippage-bps\', type=float)\n    parser.add_argument(\'--signal-threshold\', type=float)\n    parser.add_argument(\'--train-fraction\', type=float); parser.add_argument(\'--validation-fraction\', type=float)\n    parser.add_argument(\'--no-charts\', action=\'store_true\')\n    parser.add_argument(\'--render-only\', action=\'store_true\', help=\'Regenerate charts for an existing output; no search or reselection\')\n    args = parser.parse_args()\n    if args.render_only:\n        from .charts import render_results\n        render_results(args.output)\n        manifest_path = Path(args.output)/\'reproduction.json\'\n        if manifest_path.exists():\n            manifest = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n            sources = [Path(__file__).with_name(\'charts.py\')] + list(Path(__file__).with_name(\'figure_tools\').glob(\'*.py\'))\n            manifest[\'render_metadata\'] = {\n                \'versions\': {p:importlib.metadata.version(p) for p in [\'matplotlib\',\'plotly\',\'pillow\']},\n                \'code_sha256\': {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}}\n            manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n        return\n    cfg = Config(**json.loads(Path(args.config).read_text(encoding=\'utf-8\'))) if args.config else Config()\n    for name in [\'steps\',\'batch_size\',\'start\',\'end\',\'interval\',\'learning_rate\',\'max_len\',\'top_k\',\n                 \'min_entries\',\'norm_window\',\'fee_bps\',\'slippage_bps\',\'signal_threshold\',\n                 \'train_fraction\',\'validation_fraction\']:\n        if getattr(args, name) is not None: setattr(cfg, name, getattr(args, name))\n    if args.seeds: cfg.seeds = tuple(int(s.strip()) for s in args.seeds.split(\',\'))\n    if args.no_charts: cfg.charts = False\n    run(cfg, args.output, args.candles_csv, args.funding_csv)\n\n\nif __name__ == \'__main__\': main()\n', 'btc_mining/webui.py': '"""Gradio controls for the same BTC research pipeline used by the CLI."""\nimport argparse\nfrom dataclasses import asdict\nimport json\nfrom pathlib import Path\nimport shutil\nimport uuid\n\nimport gradio as gr\n\nfrom .core import Config, run\nfrom .charts import interactive_figures\n\n\ndef ui_config(start, end, steps, batch, seeds, lr, max_len, top_k, min_entries,\n              fee, slip, threshold, norm, train, validation):\n    import math\n    for value in [steps,batch,max_len,top_k,min_entries,norm]:\n        if not math.isfinite(float(value)) or not float(value).is_integer():\n            raise ValueError(\'训练步数、批大小、窗口与候选数量必须为整数\')\n    cfg = Config(start=start.strip(),end=end.strip(),steps=int(steps),batch_size=int(batch),\n        seeds=tuple(int(s.strip()) for s in seeds.split(\',\')),learning_rate=float(lr),\n        max_len=int(max_len),top_k=int(top_k),min_entries=int(min_entries),fee_bps=float(fee),\n        slippage_bps=float(slip),signal_threshold=float(threshold),norm_window=int(norm),\n        train_fraction=float(train),validation_fraction=float(validation))\n    cfg.validate()\n    return cfg\n\n\ndef train_ui(*values, progress=gr.Progress()):\n    *params, candle_csv, funding_csv = values\n    try:\n        cfg = ui_config(*params)\n        output = Path(\'btc_runs\') / uuid.uuid4().hex[:12]\n        best = run(cfg, output, candle_csv, funding_csv,\n                   progress=lambda p, text: progress(p, desc=text))\n        rows = [\'| Split | Net return | Sharpe | Max drawdown | Entries |\', \'|---|---:|---:|---:|---:|\']\n        for name in [\'train\',\'validation\',\'test\']:\n            x=best[name]\n            rows.append(f\'| {name} | {x["total_return"]:.2%} | {x["sharpe"]:.2f} | {x["max_drawdown"]:.2%} | {x["entries"]} |\')\n        status = \'测试收益为负，未通过样本外盈利验证。\' if best[\'test\'][\'total_return\'] <= 0 else \'测试收益为正；这是本次搜索结果，不保证未来收益。\'\n        plots = interactive_figures(output)\n        gallery = [(str(p), p.stem) for p in sorted((output/\'figures\').glob(\'*.png\')) if \'grayscale\' not in p.name]\n        archive = shutil.make_archive(str(output.resolve()),\'zip\',output)\n        return best[\'formula\'], \'\\n\'.join(rows), status, *plots, gallery, [archive, str(output/\'best_btc_formula.json\')]\n    except (ValueError, RuntimeError, OSError) as exc:\n        raise gr.Error(str(exc)) from exc\n\n\ndef build_ui():\n    defaults = Config()\n    with gr.Blocks(title=\'AlphaGPT BTC Factor Lab\', theme=gr.themes.Soft(primary_hue=\'blue\')) as app:\n        gr.Markdown(\'# AlphaGPT BTC Factor Lab\\nGate BTC_USDT 永续 · 1h · 多空因子挖掘 · 1 倍目标敞口\')\n        gr.Markdown(\'训练集搜索 → 验证集选择 → 测试集评估。公式、图表与数据可下载；所有日期为 UTC。\')\n        with gr.Row():\n            with gr.Column(scale=1):\n                start=gr.Textbox(label=\'开始日期 UTC\',value=defaults.start)\n                end=gr.Textbox(label=\'结束日期 UTC（不包含）\',value=defaults.end)\n                steps=gr.Slider(1,2000,value=200,step=1,label=\'每个种子的训练步数\',info=\'策略梯度更新次数\')\n                batch=gr.Dropdown([8,16,32,64,128,256,512],value=128,label=\'每步公式数量\')\n                seeds=gr.Textbox(value=\'42,43,44\',label=\'随机种子\',info=\'用英文逗号分隔\')\n                with gr.Accordion(\'搜索与交易成本\',open=False):\n                    lr=gr.Number(value=0.0003,label=\'学习率\')\n                    max_len=gr.Slider(2,12,value=12,step=1,label=\'最大公式长度\')\n                    top_k=gr.Number(value=30,precision=0,label=\'训练集候选数 Top K\')\n                    min_entries=gr.Number(value=10,precision=0,label=\'最少入场次数\')\n                    fee=gr.Number(value=5,label=\'单边手续费 bps\')\n                    slip=gr.Number(value=2,label=\'单边滑点 bps\')\n                    threshold=gr.Slider(0,0.95,value=0.2,label=\'多空信号阈值\')\n                    norm=gr.Number(value=168,precision=0,label=\'滚动归一化窗口\')\n                    train=gr.Slider(0.3,0.8,value=0.6,step=0.05,label=\'训练集比例\')\n                    validation=gr.Slider(0.1,0.4,value=0.2,step=0.05,label=\'验证集比例\')\n                with gr.Accordion(\'导入历史 CSV（可选）\',open=False):\n                    gr.Markdown(\'超过 Gate 180 天资金费率范围时，同时上传 K 线和资金费率。列格式见 README。\')\n                    candle=gr.File(label=\'K 线 CSV\',file_types=[\'.csv\'],type=\'filepath\')\n                    funding=gr.File(label=\'资金费率 CSV\',file_types=[\'.csv\'],type=\'filepath\')\n                button=gr.Button(\'开始因子挖掘\',variant=\'primary\')\n            with gr.Column(scale=3):\n                status=gr.Markdown(\'等待运行。建议先用 20 步、一个种子检查环境。\')\n                formula=gr.Textbox(label=\'验证集最佳公式\',lines=3,interactive=False)\n                metrics=gr.Markdown()\n                with gr.Tabs():\n                    with gr.Tab(\'测试净值\'): equity=gr.Plot()\n                    with gr.Tab(\'测试回撤\'): dd=gr.Plot()\n                    with gr.Tab(\'训练过程\'): training=gr.Plot()\n                    with gr.Tab(\'全部图表\'): gallery=gr.Gallery(columns=2,label=\'原始数据 / 搜索 / 测试结果\')\n                downloads=gr.Files(label=\'下载完整结果 ZIP 与公式 JSON\')\n        button.click(train_ui,inputs=[start,end,steps,batch,seeds,lr,max_len,top_k,min_entries,fee,slip,\n                    threshold,norm,train,validation,candle,funding],\n                    outputs=[formula,metrics,status,equity,dd,training,gallery,downloads],concurrency_limit=1,api_name=False)\n    return app.queue(default_concurrency_limit=1,max_size=2)\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--share\',action=\'store_true\',help=\'Create a temporary public Gradio link (for Colab)\')\n    parser.add_argument(\'--port\',type=int,default=7860)\n    args=parser.parse_args()\n    build_ui().launch(share=args.share,server_port=args.port)\n\n\nif __name__==\'__main__\': main()\n', 'btc_mining/__init__.py': '"""BTC perpetual symbolic factor mining, independent of live execution."""\n', 'btc_mining/figure_tools/export_figure.py': '"""\nscipilot-figure-skill :: export_figure.py\n=========================================\nUnified figure export to multiple formats at exact final size.\n\n- Vector preferred: PDF / SVG / EPS for line/bar/scatter (lossless, journal-friendly).\n- Raster for photos / micrographs: PNG / TIFF at >= 300 DPI; never JPEG for data figures.\n- Embeds TrueType fonts (fonttype 42) so journals don\'t reject Type-3 PDFs.\n- Optional grayscale preview to sanity-check colorblind safety.\n\nUsage\n-----\n    from export_figure import export_figure\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots()\n    ax.plot([0,1,2],[3,1,4])\n\n    paths = export_figure(\n        fig,\n        basename="figs/fig1_main",\n        formats=["pdf", "svg", "png"],\n        size_inches=(3.5, 2.625),   # 强制成 Nature 单栏尺寸\n        dpi=600,\n        grayscale_preview=True,\n    )\n    # -> [\'figs/fig1_main.pdf\', \'figs/fig1_main.svg\',\n    #     \'figs/fig1_main.png\', \'figs/fig1_main_grayscale.png\']\n\nCLI: ``python export_figure.py demo`` 生成一张演示图并多格式导出。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport sys\nfrom typing import Iterable\n\nimport matplotlib.pyplot as plt\n\n\nVECTOR_FORMATS = {"pdf", "svg", "eps"}\nRASTER_FORMATS = {"png", "tiff", "tif", "jpg", "jpeg"}\nSUPPORTED_FORMATS = VECTOR_FORMATS | RASTER_FORMATS\n\n\ndef _ensure_parent(path: str) -> None:\n    parent = os.path.dirname(os.path.abspath(path))\n    if parent and not os.path.exists(parent):\n        os.makedirs(parent, exist_ok=True)\n\n\ndef export_figure(\n    fig,\n    basename: str,\n    formats: Iterable[str] | None = None,\n    dpi: int = 300,\n    size_inches: tuple[float, float] | None = None,\n    grayscale_preview: bool = False,\n    tight: bool = True,\n    pad_inches: float = 0.05,\n    transparent: bool = False,\n) -> list[str]:\n    """\n    Export a matplotlib Figure to one or more formats at exact final size.\n\n    Args:\n        fig: matplotlib Figure object.\n        basename: 输出路径前缀（不含扩展名）；可包含子目录，自动创建。\n        formats: list/tuple of extensions, e.g. [\'pdf\', \'svg\', \'png\'].\n            Default: [\'pdf\', \'svg\', \'png\'].\n        dpi: raster 格式分辨率；建议 300（普通）/ 600（IEEE 等）。\n        size_inches: (width, height) 英寸；指定后会 fig.set_size_inches() 强制\n            最终尺寸。强烈建议传入——保证导出后不必在 Word/LaTeX 里二次缩放。\n        grayscale_preview: 额外生成一张 _grayscale.png 供色盲安全检查。\n        tight: 是否走 bbox_inches=\'tight\'（裁掉留白）。\n        pad_inches: tight 模式下保留的边距（英寸）。\n        transparent: 透明背景（PPT/海报可能需要）。\n\n    Returns:\n        实际写出的文件路径列表。\n    """\n    if formats is None:\n        formats = ("pdf", "svg", "png")\n    formats = [f.lower().lstrip(".") for f in formats]\n    unknown = [f for f in formats if f not in SUPPORTED_FORMATS]\n    if unknown:\n        raise ValueError(f"Unsupported formats: {unknown}. "\n                         f"Supported: {sorted(SUPPORTED_FORMATS)}")\n\n    if size_inches is not None:\n        if len(size_inches) != 2:\n            raise ValueError("size_inches must be (width, height)")\n        fig.set_size_inches(*size_inches)\n\n    # 强制嵌入 TrueType 字体（fonttype 42）；多家期刊明确拒绝 Type-3 PDF。\n    plt.rcParams["pdf.fonttype"] = 42\n    plt.rcParams["ps.fonttype"] = 42\n    plt.rcParams["svg.fonttype"] = "none"  # 文本仍可编辑，期刊通常更欢迎\n\n    saved: list[str] = []\n    for fmt in formats:\n        if fmt in {"jpg", "jpeg"}:\n            print(f"[scipilot-figure-skill] WARNING: skipping {fmt} — "\n                  "JPEG is lossy and unsuitable for line/text figures.",\n                  file=sys.stderr)\n            continue\n        path = f"{basename}.{fmt}"\n        _ensure_parent(path)\n        kwargs: dict = {\n            "bbox_inches": "tight" if tight else None,\n            "pad_inches": pad_inches,\n            "transparent": transparent,\n        }\n        if fmt in RASTER_FORMATS:\n            kwargs["dpi"] = dpi\n        fig.savefig(path, **kwargs)\n        saved.append(path)\n        print(f"[scipilot-figure-skill] wrote {path}")\n\n    if grayscale_preview:\n        gray_path = _grayscale_from(fig, basename, dpi=dpi)\n        if gray_path:\n            saved.append(gray_path)\n    return saved\n\n\ndef _grayscale_from(fig, basename: str, dpi: int) -> str | None:\n    """\n    导出灰度预览版用于色盲安全检查。\n    优先用 PIL 转灰度；找不到 PIL 时退化为重新画图（关闭颜色）。\n    """\n    try:\n        from PIL import Image\n    except ImportError:\n        print("[scipilot-figure-skill] Pillow not available; "\n              "grayscale preview skipped.", file=sys.stderr)\n        return None\n\n    png_path = f"{basename}.png"\n    _ensure_parent(png_path)\n    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")\n\n    gray_path = f"{basename}_grayscale.png"\n    Image.open(png_path).convert("L").save(gray_path)\n    print(f"[scipilot-figure-skill] wrote {gray_path} (grayscale preview)")\n    return gray_path\n\n\ndef _demo(out_basename: str) -> None:\n    """Generate a small demo figure for quick smoke testing."""\n    import numpy as np\n    rng = np.random.default_rng(7)\n    x = np.linspace(0, 10, 50)\n    y1 = np.sin(x) + rng.normal(0, 0.1, x.size)\n    y2 = np.cos(x) + rng.normal(0, 0.1, x.size)\n\n    fig, ax = plt.subplots(figsize=(3.5, 2.625))\n    ax.plot(x, y1, label="sin", marker="o", markersize=3)\n    ax.plot(x, y2, label="cos", marker="s", markersize=3, linestyle="--")\n    ax.set_xlabel("x")\n    ax.set_ylabel("y")\n    ax.legend(frameon=False)\n\n    paths = export_figure(\n        fig, out_basename,\n        formats=["pdf", "svg", "png"],\n        size_inches=(3.5, 2.625),\n        dpi=300,\n        grayscale_preview=True,\n    )\n    print("\\nDemo done. Files:")\n    for p in paths:\n        print(f"  {p}")\n\n\ndef _cli() -> int:\n    p = argparse.ArgumentParser(description="scipilot-figure-skill figure exporter")\n    p.add_argument("cmd", choices=["demo"], help="`demo`: 跑一张演示图导出 4 种格式")\n    p.add_argument("--out", default="./scipilot_demo",\n                   help="输出 basename (默认 ./scipilot_demo)")\n    args = p.parse_args()\n    if args.cmd == "demo":\n        _demo(args.out)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(_cli())\n', 'btc_mining/figure_tools/setup_style.py': '"""\nscipilot-figure-skill :: setup_style.py\n=======================================\nPublication-grade matplotlib / seaborn style configuration.\n\n应用出版级样式预设。支持 nature / ieee / science / general 四种期刊预设，\n支持中英文（lang=\'zh\'/\'en\'），中文模式按优先级自动查找\nNoto Sans CJK SC > Source Han Sans SC > SimHei > Microsoft YaHei\n并修正负号渲染。SciencePlots 可选——装了就用，没装回退到内置等效预设。\n\nUsage\n-----\n    from setup_style import setup_style\n\n    # Nature 单栏英文图\n    setup_style(journal=\'nature\', lang=\'en\')\n\n    # 中文期刊通用\n    setup_style(journal=\'general\', lang=\'zh\')\n\n    # 关闭 SciencePlots 强制用内置预设\n    setup_style(journal=\'ieee\', use_sciplots=False)\n\nCLI: ``python setup_style.py --list-fonts`` 列出可用 CJK 字体。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nimport warnings\n\nimport matplotlib\nimport matplotlib.font_manager as fm\nimport matplotlib.pyplot as plt\n\n\n# 期刊预设：figsize 单位为英寸，对应该期刊单栏标称宽度\n# Nature: 89mm = 3.5 in  双栏: 183mm = 7.2 in\n# Science: 与 Nature 接近\n# IEEE: 单栏 3.5 in, 双栏 7.16 in\nJOURNAL_PRESETS = {\n    "nature": {\n        "figure.figsize": (3.5, 2.625),  # 4:3 ratio @ 89mm\n        "figure.dpi": 150,\n        "savefig.dpi": 300,\n        "font.family": "sans-serif",\n        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],\n        "font.size": 7,\n        "axes.labelsize": 8,\n        "axes.titlesize": 8,\n        "xtick.labelsize": 7,\n        "ytick.labelsize": 7,\n        "legend.fontsize": 7,\n        "lines.linewidth": 1.0,\n        "lines.markersize": 4,\n        "axes.linewidth": 0.6,\n        "xtick.major.width": 0.6,\n        "ytick.major.width": 0.6,\n        "xtick.minor.width": 0.4,\n        "ytick.minor.width": 0.4,\n        "axes.spines.top": False,\n        "axes.spines.right": False,\n        "pdf.fonttype": 42,\n        "ps.fonttype": 42,\n        "svg.fonttype": "none",\n    },\n    "science": {\n        "figure.figsize": (3.5, 2.625),\n        "figure.dpi": 150,\n        "savefig.dpi": 300,\n        "font.family": "sans-serif",\n        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],\n        "font.size": 7,\n        "axes.labelsize": 7,\n        "axes.titlesize": 8,\n        "xtick.labelsize": 7,\n        "ytick.labelsize": 7,\n        "legend.fontsize": 6,\n        "lines.linewidth": 1.0,\n        "lines.markersize": 4,\n        "axes.linewidth": 0.6,\n        "axes.spines.top": False,\n        "axes.spines.right": False,\n        "pdf.fonttype": 42,\n        "ps.fonttype": 42,\n        "svg.fonttype": "none",\n    },\n    "ieee": {\n        "figure.figsize": (3.5, 2.5),\n        "figure.dpi": 150,\n        "savefig.dpi": 600,\n        "font.family": "serif",\n        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],\n        "font.size": 8,\n        "axes.labelsize": 8,\n        "axes.titlesize": 9,\n        "xtick.labelsize": 7,\n        "ytick.labelsize": 7,\n        "legend.fontsize": 7,\n        "lines.linewidth": 1.0,\n        "lines.markersize": 4,\n        "axes.linewidth": 0.7,\n        "axes.grid": False,\n        "axes.spines.top": True,\n        "axes.spines.right": True,\n        "pdf.fonttype": 42,\n        "ps.fonttype": 42,\n        "svg.fonttype": "none",\n    },\n    "general": {\n        "figure.figsize": (5.0, 3.5),\n        "figure.dpi": 150,\n        "savefig.dpi": 300,\n        "font.family": "sans-serif",\n        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],\n        "font.size": 9,\n        "axes.labelsize": 10,\n        "axes.titlesize": 11,\n        "xtick.labelsize": 9,\n        "ytick.labelsize": 9,\n        "legend.fontsize": 9,\n        "lines.linewidth": 1.2,\n        "lines.markersize": 5,\n        "axes.linewidth": 0.8,\n        "axes.spines.top": False,\n        "axes.spines.right": False,\n        "pdf.fonttype": 42,\n        "ps.fonttype": 42,\n        "svg.fonttype": "none",\n    },\n}\n\n# 中文字体优先级列表（按可用性 + 期刊接受度排序）\nCJK_FONT_PRIORITY = [\n    "Noto Sans CJK SC",\n    "Noto Sans SC",\n    "Source Han Sans SC",\n    "Source Han Sans CN",\n    "SimHei",\n    "Microsoft YaHei",\n    "PingFang SC",\n    "Heiti SC",\n    "WenQuanYi Zen Hei",\n    "Arial Unicode MS",\n]\n\nCJK_SERIF_PRIORITY = [\n    "Noto Serif CJK SC",\n    "Noto Serif SC",\n    "Source Han Serif SC",\n    "Source Han Serif CN",\n    "SimSun",\n    "STSong",\n    "Songti SC",\n]\n\nCJK_INSTALL_HINT = """\\\n找不到任何中文字体。请安装 Noto CJK 字体之一：\n\n  Linux:    sudo apt install fonts-noto-cjk    # Debian/Ubuntu\n            sudo dnf install google-noto-sans-cjk-fonts  # Fedora/RHEL\n  macOS:    brew install --cask font-noto-sans-cjk-sc\n            或下载: https://github.com/notofonts/noto-cjk/releases\n  Windows:  下载 https://github.com/notofonts/noto-cjk/releases\n            解压后右键 .ttf/.otf 文件 -> "为所有用户安装"\n\n或者列出当前已安装的中文字体：\n  python setup_style.py --list-fonts\n"""\n\n\ndef _available_fonts() -> set[str]:\n    """返回 matplotlib 已索引的全部字体名集合。"""\n    return {f.name for f in fm.fontManager.ttflist}\n\n\ndef list_cjk_fonts() -> list[str]:\n    """返回系统上可用的 CJK 字体（按优先级排序）。"""\n    available = _available_fonts()\n    hits = []\n    for f in CJK_FONT_PRIORITY + CJK_SERIF_PRIORITY:\n        if f in available and f not in hits:\n            hits.append(f)\n    # 额外做一次包含中文关键词的扫描，捕获非标准命名的中文字体\n    for f in available:\n        lower = f.lower()\n        if any(k in lower for k in ("cjk", "han", "songti", "yahei", "simhei", "simsun")):\n            if f not in hits:\n                hits.append(f)\n    return hits\n\n\ndef configure_chinese_fonts(serif_for_zh: bool = False) -> str:\n    """\n    自动检测并配置中文字体；同时修正负号渲染。\n\n    Args:\n        serif_for_zh: True 时优先选用衬线中文字体（宋体类），用于中文期刊\n            "宋体正文 + Times New Roman 数字" 的混排约定。\n    Returns:\n        实际选用的中文字体名。\n    Raises:\n        RuntimeError: 系统未安装任何识别到的中文字体。\n    """\n    available = _available_fonts()\n    priority = CJK_SERIF_PRIORITY + CJK_FONT_PRIORITY if serif_for_zh else CJK_FONT_PRIORITY\n\n    chosen = None\n    for f in priority:\n        if f in available:\n            chosen = f\n            break\n\n    if chosen is None:\n        # 兜底：扫包含 cjk/han 等关键字的字体\n        for f in available:\n            lower = f.lower()\n            if any(k in lower for k in ("cjk", "han", "song", "hei", "yahei", "kaiti")):\n                chosen = f\n                break\n\n    if chosen is None:\n        raise RuntimeError(CJK_INSTALL_HINT)\n\n    # 中文期刊常要求中文 + Times New Roman 混排：中文走中文字体，西文走 Times\n    plt.rcParams["font.family"] = ["sans-serif"] if not serif_for_zh else ["serif"]\n    if serif_for_zh:\n        plt.rcParams["font.serif"] = [chosen, "Times New Roman", "Times", "DejaVu Serif"]\n    else:\n        plt.rcParams["font.sans-serif"] = [chosen, "Arial", "Helvetica", "DejaVu Sans"]\n    # 修正负号 unicode minus 在某些中文字体里渲染成方框的问题\n    plt.rcParams["axes.unicode_minus"] = False\n    return chosen\n\n\ndef _try_sciencplots(journal: str) -> bool:\n    """If SciencePlots is installed, apply its style stack; otherwise return False."""\n    try:\n        import scienceplots  # noqa: F401\n    except ImportError:\n        return False\n\n    # SciencePlots 风格栈：基础 + 期刊变体\n    stack = ["science"]\n    if journal == "nature":\n        stack.append("nature")\n    elif journal == "ieee":\n        stack.append("ieee")\n    # 关掉 LaTeX 渲染避免环境缺 LaTeX 时崩溃；中文模式必须关\n    stack.append("no-latex")\n    try:\n        plt.style.use(stack)\n        return True\n    except OSError as e:\n        warnings.warn(f"SciencePlots style stack failed: {e}; fallback to builtin.")\n        return False\n\n\ndef setup_style(\n    journal: str = "general",\n    lang: str = "en",\n    use_sciplots: bool = True,\n    serif_for_zh: bool = False,\n    constrained_layout: bool = True,\n) -> dict:\n    """\n    应用出版级样式预设。\n\n    Args:\n        journal: \'nature\' | \'science\' | \'ieee\' | \'general\'\n        lang: \'en\' | \'zh\' — 中文模式自动配置中文字体并修正负号\n        use_sciplots: 优先尝试 SciencePlots；不可用则回退到内置预设\n        serif_for_zh: 中文模式下使用宋体类衬线字体（中文期刊常约定）\n        constrained_layout: 默认 True——全局开启 constrained_layout 自适应排版，\n            从源头减少标题/轴标签被裁、图例压数据、子图互相重叠。需要手动\n            subplots_adjust 或某些 colorbar 写法时可传 False 关闭。\n    Returns:\n        dict 包含 keys: journal / lang / sciplots_used / cjk_font / constrained_layout\n    """\n    if journal not in JOURNAL_PRESETS:\n        raise ValueError(f"Unknown journal preset: {journal}. "\n                         f"Choose from {sorted(JOURNAL_PRESETS)}")\n\n    sciplots_used = False\n    if use_sciplots:\n        sciplots_used = _try_sciencplots(journal)\n\n    # 内置预设始终在 SciencePlots 之上覆盖一遍，确保关键参数（fonttype、字号）落实\n    plt.rcParams.update(JOURNAL_PRESETS[journal])\n\n    # 默认开启自适应排版：从源头减少文字遮盖 / 裁切 / 子图重叠\n    plt.rcParams["figure.constrained_layout.use"] = constrained_layout\n\n    # 全模式默认修正负号：避免所选字体缺 U+2212 时负号渲染成方框（一种乱码）。\n    # 用 ASCII hyphen-minus 代替真减号，几乎所有字体都含，最稳妥。\n    plt.rcParams["axes.unicode_minus"] = False\n\n    cjk_font = None\n    if lang == "zh":\n        cjk_font = configure_chinese_fonts(serif_for_zh=serif_for_zh)\n    elif lang != "en":\n        raise ValueError(f"lang must be \'en\' or \'zh\', got {lang!r}")\n\n    return {\n        "journal": journal,\n        "lang": lang,\n        "sciplots_used": sciplots_used,\n        "cjk_font": cjk_font,\n        "constrained_layout": constrained_layout,\n    }\n\n\ndef _cli() -> int:\n    p = argparse.ArgumentParser(description="scipilot-figure-skill style setup")\n    p.add_argument("--list-fonts", action="store_true",\n                   help="列出当前系统上可用的 CJK 字体")\n    p.add_argument("--test", action="store_true",\n                   help="应用预设并打印 rcParams 关键值")\n    p.add_argument("--journal", default="general",\n                   choices=list(JOURNAL_PRESETS))\n    p.add_argument("--lang", default="en", choices=["en", "zh"])\n    p.add_argument("--no-sciplots", action="store_true")\n    p.add_argument("--serif-zh", action="store_true")\n    args = p.parse_args()\n\n    if args.list_fonts:\n        fonts = list_cjk_fonts()\n        if fonts:\n            print("已检测到的 CJK 字体（按优先级排序）：")\n            for f in fonts:\n                print(f"  - {f}")\n        else:\n            print("未检测到任何 CJK 字体。")\n            print(CJK_INSTALL_HINT)\n        return 0\n\n    if args.test:\n        info = setup_style(journal=args.journal, lang=args.lang,\n                           use_sciplots=not args.no_sciplots,\n                           serif_for_zh=args.serif_zh)\n        print(f"applied: {info}")\n        for k in ("figure.figsize", "font.family", "font.size",\n                  "axes.labelsize", "pdf.fonttype", "axes.unicode_minus"):\n            print(f"  {k} = {plt.rcParams[k]!r}")\n        return 0\n\n    p.print_help()\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(_cli())\n', 'btc_mining/figure_tools/visual_qa.py': '"""\nscipilot-figure-skill :: visual_qa.py\n=====================================\n出图后的「程序自检」+「渲染预览」——自检闭环的机器那一层。\n\n设计分工：\n- **程序（本脚本）** 抓**确定性**问题：缺字乱码、文字越界裁切、刻度标签重叠。\n- **AI 读图**（见 references/visual_review.md）抓**感知性**问题：图例压数据、\n  子图标签是否对齐、配色灰度可分、整体观感。\n\n两层串起来才是完整的「出图 → 渲 PNG → 程序自检 + AI 读图 → 回改 → 再看」闭环。\n\n核心能力\n--------\n- ``render_preview(fig_or_path, out_png, dpi)`` —— 渲一张中分辨率 PNG，\n  供 AI 用 Read 工具读图复核（矢量 PDF 没法直接"看像素重叠"，必须先栅格化）。\n- ``audit_layout(fig)`` —— 返回 ``[(severity, msg), ...]``：\n  * **缺字**（FAIL）：渲染时同时拦截 matplotlib 的 warnings 与 logging 两条\n    告警通道，任一报 "missing from font" 即判定成图会出方框/乱码。\n  * **文字越界裁切**（WARN）：Text 的 window_extent 超出画布边界。\n  * **刻度标签重叠**（WARN）：相邻 tick label 的包围盒水平/垂直相交。\n\nseverity 约定与 check_figure.py 保持一致：INFO < WARN < FAIL。\n\nUsage\n-----\n    from visual_qa import render_preview, audit_layout, print_report\n\n    fig, ax = plt.subplots()\n    ...\n    png = render_preview(fig, "figs/_preview.png", dpi=150)  # 给 AI 读图\n    issues = audit_layout(fig)\n    print_report(issues)\n\nCLI:\n    python visual_qa.py demo                       # 跑一遍自检演示\n    python visual_qa.py figs/fig1.png --preview out.png\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport logging\nimport os\nimport sys\nimport warnings\n\nimport matplotlib.pyplot as plt\nimport matplotlib.text as mtext\n\n\n# Windows GBK 终端下 print 中文会 UnicodeEncodeError。用 reconfigure 而非替换\n# sys.stdout：幂等、不创建新对象，多个脚本一起 import 时也不会关闭底层流。\nif sys.platform == "win32":\n    try:\n        sys.stdout.reconfigure(encoding="utf-8")\n    except (AttributeError, ValueError):\n        pass\n\nSEVERITY = {"INFO": 0, "WARN": 1, "FAIL": 2}\n_GLYPH_MARKERS = ("missing from", "Glyph", "findfont")\n\n\ndef _ensure_parent(path: str) -> None:\n    parent = os.path.dirname(os.path.abspath(path))\n    if parent and not os.path.exists(parent):\n        os.makedirs(parent, exist_ok=True)\n\n\nclass _GlyphLogHandler(logging.Handler):\n    """拦截 matplotlib logger 里关于缺字 / 找不到字体的记录。"""\n\n    def __init__(self):\n        super().__init__()\n        self.messages: list[str] = []\n\n    def emit(self, record):\n        msg = record.getMessage()\n        if any(m in msg for m in _GLYPH_MARKERS):\n            self.messages.append(msg)\n\n\ndef _draw_and_collect_glyph_warnings(fig) -> list[str]:\n    """\n    渲染一次 figure，同时从 warnings 和 logging 两条通道收集缺字告警。\n\n    matplotlib 不同版本对缺字的上报方式不一：老版本走 warnings.warn，\n    新版本走 logging。两边都挂上才不会漏。渲染顺便让 renderer 就绪，\n    供后续 window_extent 测量使用。\n    """\n    handler = _GlyphLogHandler()\n    mpl_logger = logging.getLogger("matplotlib")\n    prev_level = mpl_logger.level\n    mpl_logger.setLevel(logging.WARNING)\n    mpl_logger.addHandler(handler)\n\n    collected: list[str] = []\n    try:\n        with warnings.catch_warnings(record=True) as wlist:\n            warnings.simplefilter("always")\n            # 画到内存即可，不落盘；draw 也会触发缺字告警\n            buf = io.BytesIO()\n            fig.savefig(buf, format="png", dpi=100)\n            buf.close()\n        for w in wlist:\n            s = str(w.message)\n            if any(m in s for m in _GLYPH_MARKERS):\n                collected.append(s)\n    finally:\n        mpl_logger.removeHandler(handler)\n        mpl_logger.setLevel(prev_level)\n\n    collected.extend(handler.messages)\n    # 去重并保持顺序\n    seen, uniq = set(), []\n    for m in collected:\n        if m not in seen:\n            seen.add(m)\n            uniq.append(m)\n    return uniq\n\n\ndef _visible_texts(fig) -> list:\n    out = []\n    for t in fig.findobj(mtext.Text):\n        try:\n            if t.get_visible() and t.get_text().strip():\n                out.append(t)\n        except Exception:\n            continue\n    return out\n\n\ndef audit_layout(fig, clip_tol_px: float = 2.0, overlap_tol_px: float = 1.0\n                 ) -> list[tuple[str, str]]:\n    """\n    对一张 matplotlib Figure 做版面自检。返回 [(severity, msg), ...]。\n\n    检测项：\n        1. 缺字乱码（FAIL）—— 中文/特殊符号字体未命中。\n        2. 文字越界裁切（WARN）—— 标题/轴标签/标注超出画布。\n        3. 刻度标签重叠（WARN）—— x/y 轴相邻刻度包围盒相交。\n\n    非破坏性：只渲染测量，不修改 fig 内容。\n    """\n    issues: list[tuple[str, str]] = []\n\n    # ---- 1. 缺字（同时触发一次渲染，让 renderer 就绪）----\n    glyph_msgs = _draw_and_collect_glyph_warnings(fig)\n    # savefig temporarily changes DPI; refresh label positions at the canvas DPI\n    # before comparing their bounding boxes with the restored canvas dimensions.\n    fig.canvas.draw()\n    if glyph_msgs:\n        sample = " | ".join(glyph_msgs[:3])\n        issues.append((\n            "FAIL",\n            f"检测到缺字，成图会出现方框/乱码：{sample[:240]}。"\n            "中文图请先 setup_style(lang=\'zh\') 配置 CJK 字体；"\n            "若是负号方框，确认 axes.unicode_minus=False。"\n        ))\n\n    try:\n        renderer = fig.canvas.get_renderer()\n    except Exception:\n        fig.canvas.draw()\n        renderer = fig.canvas.get_renderer()\n\n    W = float(fig.bbox.width)\n    H = float(fig.bbox.height)\n\n    # ---- 2. 文字越界裁切 ----\n    # 刻度标签的越界由 constrained_layout / bbox_inches=\'tight\' 自动处理，且另有\n    # 刻度重叠检测覆盖——这里跳过它们，只盯 title / 轴标签 / 标注这类用户文字，\n    # 避免 constrained_layout 下刻度贴边造成的误报。\n    tick_ids = set()\n    for ax in fig.axes:\n        for tl in (*ax.get_xticklabels(), *ax.get_xticklabels(minor=True),\n                   *ax.get_yticklabels(), *ax.get_yticklabels(minor=True)):\n            tick_ids.add(id(tl))\n\n    clipped: list[str] = []\n    for t in _visible_texts(fig):\n        if id(t) in tick_ids:\n            continue\n        try:\n            bb = t.get_window_extent(renderer)\n        except Exception:\n            continue\n        if (bb.x0 < -clip_tol_px or bb.y0 < -clip_tol_px\n                or bb.x1 > W + clip_tol_px or bb.y1 > H + clip_tol_px):\n            txt = t.get_text().strip().replace("\\n", " ")\n            if txt:\n                clipped.append(txt[:24])\n    if clipped:\n        uniq = list(dict.fromkeys(clipped))[:6]\n        issues.append((\n            "WARN",\n            f"以下文字可能超出画布被裁切：{uniq}。"\n            "跑 finalize_figure(fig) 或导出时 bbox_inches=\'tight\' 兜底；"\n            "标题/标签过长可换行或缩短。"\n        ))\n\n    # ---- 3. 刻度标签重叠 ----\n    overlap_axes = 0\n    for ax in fig.axes:\n        if ax.get_subplotspec() is None:\n            continue\n        if _ticklabels_overlap(ax.get_xticklabels(), renderer,\n                               axis="x", tol=overlap_tol_px):\n            overlap_axes += 1\n            continue\n        if _ticklabels_overlap(ax.get_yticklabels(), renderer,\n                               axis="y", tol=overlap_tol_px):\n            overlap_axes += 1\n    if overlap_axes:\n        issues.append((\n            "WARN",\n            f"{overlap_axes} 个子图存在刻度标签重叠。"\n            "x 轴：ax.tick_params(axis=\'x\', rotation=30) 或减少刻度/缩短标签；"\n            "y 轴：增大子图高度或减少刻度数。"\n        ))\n\n    return issues\n\n\ndef _ticklabels_overlap(labels, renderer, axis: str, tol: float) -> bool:\n    """相邻刻度标签包围盒是否相交。axis=\'x\' 看水平、\'y\' 看垂直。"""\n    boxes = []\n    for l in labels:\n        try:\n            if l.get_visible() and l.get_text().strip():\n                boxes.append(l.get_window_extent(renderer))\n        except Exception:\n            continue\n    if len(boxes) < 2:\n        return False\n    if axis == "x":\n        boxes.sort(key=lambda b: b.x0)\n        return any(a.x1 - b.x0 > tol for a, b in zip(boxes, boxes[1:]))\n    else:\n        boxes.sort(key=lambda b: b.y0)\n        return any(a.y1 - b.y0 > tol for a, b in zip(boxes, boxes[1:]))\n\n\ndef render_preview(fig_or_path, out_png: str = "_preview.png",\n                   dpi: int = 150) -> str:\n    """\n    渲一张 PNG 预览供 AI 读图。\n\n    Args:\n        fig_or_path: matplotlib Figure 对象（自检闭环主路径），或一个已落盘\n            的图片路径（位图直接返回；PDF 需可选的 PyMuPDF）。\n        out_png: 输出 PNG 路径。\n        dpi: 预览分辨率，默认 150（够 AI 看清文字/重叠，又不至于太大）。\n\n    Returns:\n        可供 Read 读取的 PNG 路径（位图来源时返回原路径）。\n    """\n    if hasattr(fig_or_path, "savefig"):\n        _ensure_parent(out_png)\n        fig_or_path.savefig(out_png, dpi=dpi, bbox_inches="tight")\n        return out_png\n\n    path = str(fig_or_path)\n    if not os.path.exists(path):\n        raise FileNotFoundError(path)\n    ext = os.path.splitext(path)[1].lower().lstrip(".")\n    if ext in {"png", "tif", "tiff", "jpg", "jpeg", "bmp"}:\n        return path\n    if ext == "pdf":\n        try:\n            import fitz  # PyMuPDF, 可选\n        except ImportError as e:\n            raise RuntimeError(\n                "把 PDF 渲成预览需要 PyMuPDF（pip install pymupdf）。"\n                "自检闭环更推荐直接把 matplotlib Figure 对象传给 render_preview，"\n                "在导出前就完成读图复核。"\n            ) from e\n        doc = fitz.open(path)\n        pix = doc[0].get_pixmap(dpi=dpi)\n        _ensure_parent(out_png)\n        pix.save(out_png)\n        doc.close()\n        return out_png\n    raise RuntimeError(f"不支持从 .{ext} 生成预览；请传 Figure 对象或位图。")\n\n\ndef print_report(issues: list[tuple[str, str]]) -> str:\n    """打印 audit_layout 的结果，返回 verdict（PASS/WARN/FAIL）。"""\n    if not issues:\n        print("  [PASS] 程序自检未发现缺字 / 裁切 / 刻度重叠。")\n        print("  >>> 仍需 AI 读图复核感知性问题（见 visual_review.md）。")\n        return "PASS"\n    max_sev = max(SEVERITY[s] for s, _ in issues)\n    verdict = {2: "FAIL", 1: "WARN", 0: "INFO"}[max_sev]\n    for sev, msg in sorted(issues, key=lambda x: -SEVERITY[x[0]]):\n        print(f"  [{sev}] {msg}")\n    print(f"  >>> verdict: {verdict}（修完再渲一次 PNG 让 AI 读图复核）")\n    return verdict\n\n\ndef _demo() -> int:\n    """构造一张有版面问题的图，演示 audit_layout 能抓出来。"""\n    import numpy as np\n    rng = np.random.default_rng(1)\n\n    fig, ax = plt.subplots(figsize=(3.0, 2.2))\n    cats = [f"very_long_condition_name_{i}" for i in range(12)]\n    ax.bar(range(12), rng.random(12))\n    ax.set_xticks(range(12))\n    ax.set_xticklabels(cats)  # 故意不旋转 -> x 轴刻度必然重叠\n    ax.set_title("An intentionally overlong title that runs off the canvas edge")\n    ax.set_ylabel("value")\n\n    print("=== visual_qa demo：对一张刻意做坏版面的图自检 ===")\n    issues = audit_layout(fig)\n    print_report(issues)\n    out = render_preview(fig, "./visual_qa_demo.png", dpi=120)\n    print(f"\\n预览已写出：{out}（可用 Read 工具读图复核）")\n    print("注：缺字检测只在中文未配字体等情况下才会 FAIL；本 demo 主要展示裁切+重叠。")\n    return 0\n\n\ndef _cli() -> int:\n    p = argparse.ArgumentParser(description="scipilot-figure-skill visual QA")\n    p.add_argument("target", nargs="?", help="图片路径；或 \'demo\'")\n    p.add_argument("--preview", metavar="OUT.png",\n                   help="把 target 渲成 PNG 预览到此路径")\n    p.add_argument("--dpi", type=int, default=150)\n    args = p.parse_args()\n\n    if args.target == "demo" or args.target is None:\n        return _demo()\n\n    if args.preview:\n        out = render_preview(args.target, args.preview, dpi=args.dpi)\n        print(f"[visual_qa] 预览：{out}")\n    else:\n        # 对已落盘文件，audit_layout 需要 Figure 对象，这里只能提示\n        print("[visual_qa] 对已落盘图片只支持 --preview 渲图；"\n              "版面自检 audit_layout 请在画图脚本里对 Figure 对象调用。")\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(_cli())\n', 'btc_mining/figure_tools/__init__.py': '"""Bundled figure skill utilities; copied unchanged for portable Colab exports."""\n'}
for name, source in FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(ROOT / "colab"))
sys.path.insert(0, str(ROOT))
print("Sources ready:", ROOT)


## 2. 离线正确性检查
合成数据只用于测试，不会被当作 BTC 结果。

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "colab"), "-p", "test_btc_colab.py"], cwd=ROOT, check=True)


## 3. 配置本次搜索

默认获取最近 170 天已结束的 K 线，日期和数据哈希会记录在结果中。
实测 Gate 资金费率接口拒绝超过 180 天的历史；更长范围需自行提供真实历史 CSV。
至少保留足够历史，使训练、验证、测试各有 100 根以上可交易 K 线。
默认 3 个种子，每个 200 步；先用 `steps=20, seeds=(42,)` 检查运行，再恢复默认正式搜索。
手续费与滑点是假设值，请按自己的账户和成交方式调整。重复看测试集再调参会破坏其独立性。


In [ ]:
from btc_colab import Config, run
cfg = Config(
    contract="BTC_USDT", interval="1h",
    # start/end 默认最近170天（UTC）；也可填写180天以内的固定日期。
    steps=200, batch_size=128, seeds=(42, 43, 44),
    fee_bps=5.0, slippage_bps=2.0,
    signal_threshold=0.2, top_k=30,
)
OUTPUT = ROOT / "btc_results"
cfg.validate()
print(cfg)


## 4. 获取 Gate 数据并搜索

完整性检查失败会停止；不会用零资金费率或前向填充价格掩盖数据问题。
若 Colab 网络无法访问 Gate，可上传同市场的两个 CSV：
`time,open,high,low,close,volume`（volume 为 USDT 成交额）和 `time,rate`（真实资金费率记录）。
`time` 都为 Unix 秒，再传给 `run(..., candles_csv=..., funding_csv=...)`。


In [ ]:
best = run(cfg, output_dir=OUTPUT)


## 5. 查看与下载最佳公式
公式 JSON 同时包含特征、算子、信号、成本、时间切分与测试结果。BTC token 不能直接交给原 Solana runner。

In [ ]:
import pandas as pd
print((OUTPUT / "best_btc_formula.txt").read_text(encoding="utf-8"))
display(pd.DataFrame({k: best[k] for k in ["train", "validation", "test"]}).T)
display(pd.read_csv(OUTPUT / "candidates.csv").head(10))


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(ROOT / "btc_formula_results"), "zip", OUTPUT)
files.download(archive)
# 压缩包包括最佳公式 JSON/TXT、候选排名、训练日志、测试收益轨迹、原始数据与复现信息。
